In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, copy, random, time, threading, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import itertools

In [ ]:
DATA_PATH = "/content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt"

BASE_MODEL_DIR = "/content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel"
BASE_CKPT      = os.path.join(BASE_MODEL_DIR, "elec_base_best.pth")

NOV_PROJECT_DIR = "/content/drive/MyDrive/298/Elec/NovTest/"

PROJECT_DIR = NOV_PROJECT_DIR

STUDENTS_DIR       = os.path.join(PROJECT_DIR, "students_nov")      # will hold novel students 5–8
STUDENTS_ADV_DIR   = os.path.join(PROJECT_DIR, "students_adv_nov")  # adv trained novel students 5–8
RESULTS_DIR        = os.path.join(PROJECT_DIR, "results")
PLOTS_DIR          = os.path.join(RESULTS_DIR, "plots")
ARTIFACTS_DIR      = os.path.join(RESULTS_DIR, "artifacts")
MODEL_SIZES_CSV    = os.path.join(RESULTS_DIR, "model_sizes_nov.csv")

for d in [PROJECT_DIR, STUDENTS_DIR, STUDENTS_ADV_DIR, RESULTS_DIR, PLOTS_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("NOV_PROJECT_DIR:", NOV_PROJECT_DIR)
print("STUDENTS_DIR:", STUDENTS_DIR)
print("STUDENTS_ADV_DIR:", STUDENTS_ADV_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

NOV_PROJECT_DIR: /content/drive/MyDrive/298/Elec/NovTest/
STUDENTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/students_nov
STUDENTS_ADV_DIR: /content/drive/MyDrive/298/Elec/NovTest/students_adv_nov
RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/results


In [ ]:
FREQ = "15min"
STEPS_PER_HOUR = 4
HOURS_PER_DAY  = 24

HIST_HOURS  = 96
HZN_HOURS   = 24
HIST_STEPS  = HIST_HOURS * STEPS_PER_HOUR   # 384
HZN_STEPS   = HZN_HOURS  * STEPS_PER_HOUR   # 96
SLIDE       = 1

TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.10

BATCH       = 16
PATCH       = 4
D_FEATS     = 1
D_MODEL     = 256
N_HEAD      = 4
N_LAYERS    = 3
DIM_FF      = 512
DROPOUT     = 0.1

LR_BASE     = 5e-4
LR_STUDENT  = 1e-4
EPOCHS_BASE = 30
EPOCHS_STU  = 5

FGSM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]

RFGSM_EVAL_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
RFGSM_ALPHA_FACTOR  = 1.0

N_RUNS = 30

HEARTBEAT_SECS = 600  # 10 minutes

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

Device: cuda


In [ ]:
# utilities

def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t

def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

def append_df_to_csv(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

def model_n_params(model: nn.Module):
    return sum(p.numel() for p in model.parameters())

def model_file_size_mb(path: str):
    if not os.path.exists(path):
        return None
    return os.path.getsize(path) / (1024.0 * 1024.0)

In [ ]:
# data loading

print("Reading raw txt:", DATA_PATH)
df = pd.read_csv(
    DATA_PATH,
    sep=';',
    quotechar='"',
    decimal=',',
    parse_dates=[0],
    index_col=0,
    na_values=['', 'NA', 'NaN'],
    low_memory=False
)
df.index.name = "timestamp"
df = df.sort_index()
print("Raw shape:", df.shape)

METER = "MT_001"
print("Using single meter:", METER)

# restrict to 2013 only
START_DT = pd.Timestamp("2013-01-01 00:00:00")
END_DT   = pd.Timestamp("2013-12-31 23:45:00")

df = df[[METER]]
df = df.loc[(df.index >= START_DT) & (df.index <= END_DT)].copy()

# regularize index to strict 15 min grid
full_idx = pd.date_range(START_DT, END_DT, freq=FREQ)
df = df.reindex(full_idx)

# fill up to 4 consecutive missing steps forward & backward
df = df.apply(pd.to_numeric, errors='coerce')
df = df.ffill(limit=4).bfill(limit=4)
print("Post filtering shape:", df.shape)

# scale on early TRAIN_FRAC portion of time
n_t       = len(df)
split_idx = int(TRAIN_FRAC * n_t)
df_train  = df.iloc[:split_idx]
df_all    = df

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(df_train),
    index=df_train.index,
    columns=df_train.columns
)
full_scaled = pd.DataFrame(
    scaler.transform(df_all),
    index=df_all.index,
    columns=df_all.columns
)
print("Scaled shape:", full_scaled.shape)

Reading raw txt: /content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt
Raw shape: (140256, 370)
Using single meter: MT_001
Post filtering shape: (35040, 1)
Scaled shape: (35040, 1)


In [ ]:
# windowing

values = full_scaled.values.astype(np.float32)  # (T, 1)
T_len, D = values.shape
assert D == 1, f"Expected 1 feature, got {D}"
print("T, D:", T_len, D)

X_list, Y_list, meta = [], [], []
for t in range(HIST_STEPS, T_len - HZN_STEPS, SLIDE):
    x = values[t - HIST_STEPS:t, :]      # (384, 1)
    y = values[t:t + HZN_STEPS, :]       # (96, 1)
    X_list.append(x)
    Y_list.append(y)
    meta.append((
        full_scaled.index[t - HIST_STEPS],
        full_scaled.index[t - 1],
        full_scaled.index[t],
        full_scaled.index[t + HZN_STEPS - 1],
    ))

X = np.stack(X_list)   # (N, 384, 1)
Y = np.stack(Y_list)   # (N, 96, 1)
meta_df = pd.DataFrame(meta, columns=["x_start", "x_end", "y_start", "y_end"])
print("Windowed tensors:", X.shape, Y.shape)

T, D: 35040 1
Windowed tensors: (34560, 384, 1) (34560, 96, 1)


In [ ]:
# chronological split

N = len(X)
n_train = int(TRAIN_FRAC * N)
n_val   = int(VAL_FRAC * N)
n_test  = N - n_train - n_val

idx_train = slice(0, n_train)
idx_val   = slice(n_train, n_train + n_val)
idx_test  = slice(n_train + n_val, N)

X_train, Y_train = X[idx_train], Y[idx_train]
X_val,   Y_val   = X[idx_val],   Y[idx_val]
X_test,  Y_test  = X[idx_test],  Y[idx_test]

print("Splits:")
print("  train:", X_train.shape)
print("  val  :", X_val.shape)
print("  test :", X_test.shape)

np.save(os.path.join(ARTIFACTS_DIR, "X_test.npy"), X_test)
np.save(os.path.join(ARTIFACTS_DIR, "Y_test.npy"), Y_test)
print("Saved full test X/Y arrays to ARTIFACTS_DIR.")

Splits:
  train: (24192, 384, 1)
  val  : (3456, 384, 1)
  test : (6912, 384, 1)
Saved full test X/Y arrays to ARTIFACTS_DIR.


In [ ]:
# data loaders

class ElecSeq2SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)  # (N, hist, D)
        self.Y = torch.from_numpy(Y)  # (N, hzn, D)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

train_dl = DataLoader(
    ElecSeq2SeqDataset(X_train, Y_train),
    batch_size=BATCH, shuffle=True,
    num_workers=2, pin_memory=True
)
val_dl = DataLoader(
    ElecSeq2SeqDataset(X_val, Y_val),
    batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True
)
test_dl = DataLoader(
    ElecSeq2SeqDataset(X_test, Y_test),
    batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True
)

In [ ]:
# patching

def patchify(x, patch):
    # x: (B, T, D) -> (B, T//patch, D) by mean over non-overlap windows
    if patch == 1:
        return x
    B, T, D = x.shape
    T_trim = (T // patch) * patch
    x = x[:, :T_trim, :].reshape(B, T_trim // patch, patch, D).mean(dim=2)
    return x

def unpatchify(y_patch, patch, T_out):
    # y_patch: (B, Tp, D) -> (B, T_out, D) by repeat
    if patch == 1:
        return y_patch[:, :T_out, :]
    y = y_patch.repeat_interleave(patch, dim=1)
    return y[:, :T_out, :]

In [ ]:
# transformer

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=6000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        return x + self.pe[:, :x.size(1), :]



class ElecSeq2SeqTransformer(nn.Module):
    def __init__(self,
                 d_feats,
                 d_model=256,
                 nhead=4,
                 num_layers=3,
                 dim_ff=512,
                 dropout=0.1,
                 hist_steps=HIST_STEPS,
                 hzn_steps=HZN_STEPS,
                 patch=PATCH):
        super().__init__()
        self.d_feats    = d_feats
        self.d_model    = d_model
        self.hist_steps = hist_steps
        self.hzn_steps  = hzn_steps
        self.patch      = patch

        self.in_proj = nn.Linear(d_feats, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.enc_pos = PositionalEncoding(d_model, max_len=(hist_steps // patch) + 16)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.dec_pos = PositionalEncoding(d_model, max_len=(hzn_steps // patch) + 16)

        self.out_proj = nn.Linear(d_model, d_feats)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        # x: (B, hist_steps, d_feats)
        B, T, D = x.shape
        x_p = patchify(x, self.patch)          # (B, T_p, D)
        z   = self.in_proj(x_p)                # (B, T_p, d_model)
        z   = self.enc_pos(z)
        mem = self.encoder(z)                  # (B, T_p, d_model)

        dec_Tp = self.hzn_steps // self.patch
        tgt = torch.zeros(B, dec_Tp, self.d_model, device=x.device)
        tgt = self.dec_pos(tgt)
        out = self.decoder(tgt, mem)           # (B, dec_Tp, d_model)
        y_p = self.out_proj(out)               # (B, dec_Tp, d_feats)
        y   = unpatchify(y_p, self.patch, self.hzn_steps)
        return y

In [ ]:
# losses

mse_loss = nn.MSELoss()

def rmse_only(pred, true):
    mse = mse_loss(pred, true)
    return torch.sqrt(mse + 1e-12)

def seq_rmse_np(y_true_np, y_pred_np):
    return float(np.sqrt(mean_squared_error(y_true_np.ravel(), y_pred_np.ravel())))

In [ ]:
# fgsm attacks

def fgsm_attack(attacker_model, X, Y, eps, return_grad_norm=False):
    """
    Single-step FGSM with anti-masking safeguards.
    """
    attacker_model.eval()
    X_adv = X.clone().detach().requires_grad_(True)

    pred = attacker_model(X_adv)
    loss = mse_loss(pred, Y)

    grad = torch.autograd.grad(loss, X_adv, only_inputs=True, retain_graph=False)[0]
    grad_sign = grad.sign()

    X_pert = (X_adv + eps * grad_sign).clamp(0.0, 1.0)

    if return_grad_norm:
        gn = grad.view(grad.size(0), -1).norm(p=2, dim=1)
        return X_pert.detach(), gn.detach()
    return X_pert.detach()

def rfgsm_attack(attacker_model, X, Y, eps, alpha=None, return_grad_norm=False):
    """
    Random start FGSM (rFGSM) with anti-masking safeguards.
    """
    attacker_model.eval()

    if alpha is None:
        alpha = eps * RFGSM_ALPHA_FACTOR

    noise = torch.empty_like(X).uniform_(-eps, eps)
    X0 = (X + noise).clamp(0.0, 1.0)
    X0 = X0.detach().requires_grad_(True)

    pred = attacker_model(X0)
    loss = mse_loss(pred, Y)

    grad = torch.autograd.grad(loss, X0, only_inputs=True, retain_graph=False)[0]
    grad_sign = grad.sign()

    X_adv = (X0 + alpha * grad_sign).clamp(0.0, 1.0)

    if return_grad_norm:
        gn = grad.view(grad.size(0), -1).norm(p=2, dim=1)
        return X_adv.detach(), gn.detach()
    return X_adv.detach()

In [ ]:
# base model training

def train_base(model, train_loader, val_loader,
               epochs=EPOCHS_BASE, lr=LR_BASE,
               ckpt_dir=BASE_MODEL_DIR,
               results_dir=RESULTS_DIR,
               model_sizes_csv=MODEL_SIZES_CSV):
    os.makedirs(ckpt_dir, exist_ok=True)
    opt = optim.AdamW(model.parameters(), lr=lr)
    best      = float("inf")
    best_path = os.path.join(ckpt_dir, "elec_base_best.pth")

    history = []

    hb_flag, hb_thread = start_heartbeat("base_train")
    start_time = time.time()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot = cnt = 0
            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = rmse_only(pred, yb)

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot += loss.item() * bs
                cnt += bs
            train_loss = tot / max(1, cnt)

            # Val
            model.eval()
            vt = vc = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(device)
                    yb = yb.to(device)
                    pred = model(xb)
                    loss = rmse_only(pred, yb)
                    vt += loss.item() * xb.size(0)
                    vc += xb.size(0)
            val_loss = vt / max(1, vc)

            elapsed = (time.time() - start_time) / 60.0
            print(f"[Base] Epoch {ep:02d} | train {train_loss:.4f} | val {val_loss:.4f} | elapsed {elapsed:.2f} min")

            if val_loss < best:
                best = val_loss
                torch.save(model.state_dict(), best_path)
                print("  [Base] Saved best to", best_path, flush=True)

            history.append({
                "epoch": ep,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "elapsed_min": elapsed,
            })

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    base_history_csv = os.path.join(results_dir, "base_train_history_nov.csv")
    df_hist = pd.DataFrame(history)
    df_hist.to_csv(base_history_csv, index=False)
    print("Saved base training history to:", base_history_csv)

    n_params = model_n_params(model)
    size_mb  = model_file_size_mb(best_path)
    df_ms = pd.DataFrame([{
        "model_name": "base_transformer_nov",
        "path": best_path,
        "params": n_params,
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, model_sizes_csv)

    return best_path

In [ ]:
#train or load base model

if os.path.exists(BASE_CKPT):
    print("\nFound existing base checkpoint:", BASE_CKPT)
else:
    print("\nBase checkpoint not found, training base model now...")
    base_tmp = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    BASE_CKPT = train_base(base_tmp, train_dl, val_dl)
    print("Base trained and saved to:", BASE_CKPT)

# Reload best base model on CPU for student generation
base_cpu = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to("cpu")
base_cpu.load_state_dict(torch.load(BASE_CKPT, map_location="cpu"))
print("Loaded base model into CPU for student generation.")


Found existing base checkpoint: /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loaded base model into CPU for student generation.


In [ ]:
# novel student parameter selection

def is_enc_self_attn_param(name: str) -> bool:
    return name.startswith("encoder.layers.") and ".self_attn." in name

def is_dec_attn_param(name: str) -> bool:
    if not name.startswith("decoder.layers."):
        return False
    return (".self_attn." in name) or (".multihead_attn." in name)

def is_ffn_param(name: str) -> bool:
    if not (name.startswith("encoder.layers.") or name.startswith("decoder.layers.")):
        return False
    return (".linear1." in name) or (".linear2." in name)

def is_norm_param(name: str) -> bool:
    if not (name.startswith("encoder.layers.") or name.startswith("decoder.layers.")):
        return False
    return ".norm" in name   # catches norm1, norm2, norm3, etc.

novel_student_specs = [
    dict(
        idx=5,
        name="student_05_enc_attn",
        selector=is_enc_self_attn_param,
        noise_scale=1e-3,
    ),
    dict(
        idx=6,
        name="student_06_dec_attn",
        selector=is_dec_attn_param,
        noise_scale=1e-3,
    ),
    dict(
        idx=7,
        name="student_07_ffn",
        selector=is_ffn_param,
        noise_scale=1e-3,
    ),
    dict(
        idx=8,
        name="student_08_norms",
        selector=is_norm_param,
        noise_scale=1e-3,
    ),
]

In [ ]:
# generate novel students

novel_student_paths = []

print("\n[Novel] creating students 5–8 from base")
for spec in novel_student_specs:
    idx       = spec["idx"]
    name      = spec["name"]
    selector  = spec["selector"]
    sigma     = spec["noise_scale"]

    print(f"  [Novel Student {idx}] pattern={name}, noise_scale={sigma}")

    st = copy.deepcopy(base_cpu)

    with torch.no_grad():
        for pname, p in st.named_parameters():
            if selector(pname):
                p.add_(sigma * torch.randn_like(p))

    outp = os.path.join(STUDENTS_DIR, f"{name}.pth")
    torch.save(st.state_dict(), outp)
    novel_student_paths.append((idx, name, outp))
    print(f"    saved novel student {idx} to:", outp)

    size_mb = model_file_size_mb(outp)
    df_ms = pd.DataFrame([{
        "model_name": name,
        "path": outp,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, MODEL_SIZES_CSV)

print("\n[Novel] students created:")
for idx, name, path in novel_student_paths:
    print(f"  {idx}: {name} - {path}")


[Novel] creating students 5–8 from base
  [Novel Student 5] pattern=student_05_enc_attn, noise_scale=0.001
    saved novel student 5 to: /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_05_enc_attn.pth
  [Novel Student 6] pattern=student_06_dec_attn, noise_scale=0.001
    saved novel student 6 to: /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_06_dec_attn.pth
  [Novel Student 7] pattern=student_07_ffn, noise_scale=0.001
    saved novel student 7 to: /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_07_ffn.pth
  [Novel Student 8] pattern=student_08_norms, noise_scale=0.001
    saved novel student 8 to: /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_08_norms.pth

[Novel] students created:
  5: student_05_enc_attn - /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_05_enc_attn.pth
  6: student_06_dec_attn - /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_06_dec_attn.pth
  7: student_07_ffn - /content/drive/MyDrive

In [ ]:
novel_adv_student_paths = []

print("\n[Novel] adversarial training of students 5 to 8")

for idx, name, sp in novel_student_paths:
    print(f"\n[Novel Student {idx} | {name}] adversarial training...")

    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))

    opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)

    stu_history = []

    hb_flag, hb_thread = start_heartbeat(f"novel_student_{idx}_fgsm_train")
    start_time = time.time()

    try:
        for ep in range(1, EPOCHS_STU + 1):
            stu.train()
            tot = cnt = 0

            for xb, yb in train_dl:
                xb = xb.to(device)
                yb = yb.to(device)

                # clean loss
                pred_c = stu(xb)
                Lc = rmse_only(pred_c, yb)

                # multi epsilon FGSM adversarial loss
                adv_losses = []
                for eps in FGSM_TRAIN_EPS_LIST:
                    xa = fgsm_attack(stu, xb, yb, eps=eps)
                    pred_a = stu(xa)
                    La = rmse_only(pred_a, yb)
                    adv_losses.append(La)

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                loss = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot += loss.item() * bs
                cnt += bs

            avg_loss = tot / max(1, cnt)
            elapsed  = (time.time() - start_time) / 60.0
            print(f"  [Novel Student {idx}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} | elapsed={elapsed:.2f} min")

            stu_history.append({
                "student_idx": int(idx),
                "student_name": name,
                "epoch": ep,
                "avg_loss": avg_loss,
                "elapsed_min": elapsed,
            })

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    stu_hist_path = os.path.join(RESULTS_DIR, f"{name}_fgsm_train_history.csv")
    df_stu_hist = pd.DataFrame(stu_history)
    df_stu_hist.to_csv(stu_hist_path, index=False)
    print(f"  [Novel Student {idx}] Saved training history to:", stu_hist_path)

    outp_adv = os.path.join(STUDENTS_ADV_DIR, f"{name}_rfgsm_adv.pth")
    torch.save(stu.to("cpu").state_dict(), outp_adv)
    novel_adv_student_paths.append((idx, name, outp_adv))
    print(f"  [Novel Student {idx}] Saved adv student:", outp_adv)

    size_mb = model_file_size_mb(outp_adv)
    df_ms = pd.DataFrame([{
        "model_name": f"{name}_rfgsm_adv",
        "path": outp_adv,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, MODEL_SIZES_CSV)

novel_adv_student_paths = sorted(novel_adv_student_paths, key=lambda x: x[0])

print("\n[Novel] adv-trained student paths:")
for idx, name, path in novel_adv_student_paths:
    print(f"  {idx}: {name}_rfgsm_adv -> {path}")


[Novel] adversarial training of students 5 to 8

[Novel Student 5 | student_05_enc_attn] adversarial training...
  [Novel Student 5] Epoch 01 | avg_loss=0.1033 | elapsed=3.97 min
  [Novel Student 5] Epoch 02 | avg_loss=0.0970 | elapsed=7.96 min
[heartbeat:novel_student_5_fgsm_train] running... elapsed=10.00 min
  [Novel Student 5] Epoch 03 | avg_loss=0.0943 | elapsed=11.99 min
  [Novel Student 5] Epoch 04 | avg_loss=0.0926 | elapsed=15.94 min
  [Novel Student 5] Epoch 05 | avg_loss=0.0911 | elapsed=19.89 min
  [Novel Student 5] Saved training history to: /content/drive/MyDrive/298/Elec/NovTest/results/student_05_enc_attn_fgsm_train_history.csv
  [Novel Student 5] Saved adv student: /content/drive/MyDrive/298/Elec/NovTest/students_adv_nov/student_05_enc_attn_rfgsm_adv.pth

[Novel Student 6 | student_06_dec_attn] adversarial training...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  [Novel Student 6] Epoch 01 | avg_loss=0.1037 | elapsed=3.97 min
  [Novel Student 6] Epoch 02 | avg_loss=0.0969 | elapsed=7.92 min
[heartbeat:novel_student_6_fgsm_train] running... elapsed=10.00 min
  [Novel Student 6] Epoch 03 | avg_loss=0.0940 | elapsed=11.88 min
  [Novel Student 6] Epoch 04 | avg_loss=0.0925 | elapsed=15.85 min
  [Novel Student 6] Epoch 05 | avg_loss=0.0914 | elapsed=19.83 min
  [Novel Student 6] Saved training history to: /content/drive/MyDrive/298/Elec/NovTest/results/student_06_dec_attn_fgsm_train_history.csv
  [Novel Student 6] Saved adv student: /content/drive/MyDrive/298/Elec/NovTest/students_adv_nov/student_06_dec_attn_rfgsm_adv.pth

[Novel Student 7 | student_07_ffn] adversarial training...
  [Novel Student 7] Epoch 01 | avg_loss=0.1034 | elapsed=3.95 min
  [Novel Student 7] Epoch 02 | avg_loss=0.0969 | elapsed=7.94 min
[heartbeat:novel_student_7_fgsm_train] running... elapsed=10.00 min
  [Novel Student 7] Epoch 03 | avg_loss=0.0942 | elapsed=11.92 min
  [N

In [ ]:
NOV_FGSM_RUNS_CSV        = os.path.join(RESULTS_DIR, "nov_fgsm_morphence_runs.csv")
NOV_FGSM_STATS_CSV       = os.path.join(RESULTS_DIR, "nov_fgsm_morphence_stats.csv")
NOV_XFER_MATRIX_CSV      = os.path.join(RESULTS_DIR, "nov_fgsm_transfer_matrix.csv")
NOV_WEIGHT_DIVERSITY_CSV = os.path.join(RESULTS_DIR, "nov_fgsm_weight_diversity.csv")
NOV_PRED_DIVERSITY_CSV   = os.path.join(RESULTS_DIR, "nov_fgsm_prediction_diversity.csv")
NOV_SINGLE_RUN_CSV       = os.path.join(RESULTS_DIR, "nov_fgsm_morphence_single_run.csv")

print("\n[NovEval] RESULTS_DIR:", RESULTS_DIR)
print("[NovEval] NOV_FGSM_RUNS_CSV:", NOV_FGSM_RUNS_CSV)


[NovEval] RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/results
[NovEval] NOV_FGSM_RUNS_CSV: /content/drive/MyDrive/298/Elec/NovTest/results/nov_fgsm_morphence_runs.csv


In [ ]:
best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(BASE_CKPT, map_location=device))
best_base.eval()
print("[NovEval] Loaded base model from:", BASE_CKPT)

[NovEval] Loaded base model from: /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth


In [ ]:
adv_student_paths_nov = sorted(
    [
        os.path.join(STUDENTS_ADV_DIR, f)
        for f in os.listdir(STUDENTS_ADV_DIR)
        if f.endswith("_rfgsm_adv.pth")
    ]
)

assert len(adv_student_paths_nov) > 0, "[NovEval] No *_rfgsm_adv.pth found in students_adv_nov dir!"
print("[NovEval] Novel adv students:")
for p in adv_student_paths_nov:
    print("  ", p)

adv_students_nov = []
for sp in adv_student_paths_nov:
    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))
    stu.eval()
    adv_students_nov.append(stu)

student_names_nov = [os.path.basename(p).replace(".pth", "") for p in adv_student_paths_nov]
print(f"[NovEval] Loaded {len(adv_students_nov)} novel adversarially-trained students.")

[NovEval] Novel adv students:
   /content/drive/MyDrive/298/Elec/NovTest/students_adv_nov/student_05_enc_attn_rfgsm_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/students_adv_nov/student_06_dec_attn_rfgsm_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/students_adv_nov/student_07_ffn_rfgsm_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/students_adv_nov/student_08_norms_rfgsm_adv.pth
[NovEval] Loaded 4 novel adversarially-trained students.


In [ ]:
@torch.no_grad()
def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        p  = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

@torch.no_grad()
def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        preds = []
        for m in models:
            preds.append(m(xb))
        preds = torch.stack(preds, dim=0).mean(dim=0)  # (B,H,D)
        ys.append(yb.cpu().numpy())
        ps.append(preds.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_pair_rfgsm(defender, attacker, loader, eps, alpha=None):
    defender.eval()
    attacker.eval()
    if alpha is None:
        alpha = eps * RFGSM_ALPHA_FACTOR

    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        xa = rfgsm_attack(attacker, xb, yb, eps=eps, alpha=alpha)
        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_random_switch_rfgsm(models, loader, eps, alpha=None):
    if alpha is None:
        alpha = eps * RFGSM_ALPHA_FACTOR

    for m in models:
        m.eval()

    ys, ps = [], []
    n_stu = len(models)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        atk_idx  = random.randrange(n_stu)
        def_idx  = random.randrange(n_stu)
        attacker = models[atk_idx]
        defender = models[def_idx]

        xa = rfgsm_attack(attacker, xb, yb, eps=eps, alpha=alpha)
        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
# diversity metrics

def flatten_params(model):
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])

def compute_weight_diversity_nov(students, names, out_csv=NOV_WEIGHT_DIVERSITY_CSV):
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)
    rows = []
    for i, j in itertools.combinations(range(k), 2):
        d = torch.norm(vecs[i] - vecs[j], p=2).item()
        rows.append({
            "student_i": names[i],
            "student_j": names[j],
            "l2_distance": d,
        })
    df_w = pd.DataFrame(rows)
    df_w.to_csv(out_csv, index=False)
    print("[NovEval] Saved weight diversity to:", out_csv)
    return df_w

@torch.no_grad()
def compute_prediction_diversity_nov(students, loader, out_csv=NOV_PRED_DIVERSITY_CSV):
    for m in students:
        m.eval()

    k = len(students)
    H = HZN_STEPS

    sum_pred  = np.zeros(H, dtype=np.float64)
    sum_pred2 = np.zeros(H, dtype=np.float64)
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device)
        preds_stu = []
        for m in students:
            preds_stu.append(m(xb).cpu().numpy())  # (B,H,1)
        preds_stu = np.stack(preds_stu, axis=0)  # (k,B,H,1)

        # mean prediction per student over batch, then average over students
        preds_mean_batch = preds_stu.mean(axis=1).squeeze(-1)  # (k,H)

        sum_pred  += preds_mean_batch.mean(axis=0)
        sum_pred2 += (preds_mean_batch ** 2).mean(axis=0)
        count     += 1

    mean_pred  = sum_pred / max(1, count)
    mean_pred2 = sum_pred2 / max(1, count)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "horizon_idx": np.arange(H),
        "mean_pred": mean_pred,
        "var_pred": var_pred,
    })
    df_p.to_csv(out_csv, index=False)
    print("[NovEval] Saved prediction diversity to:", out_csv)
    return df_p

print("\n[NovEval] Computing diversity metrics for novel students...")
_ = compute_weight_diversity_nov(adv_students_nov, student_names_nov)
_ = compute_prediction_diversity_nov(adv_students_nov, test_dl)


[NovEval] Computing diversity metrics for novel students...
[NovEval] Saved weight diversity to: /content/drive/MyDrive/298/Elec/NovTest/results/nov_fgsm_weight_diversity.csv
[NovEval] Saved prediction diversity to: /content/drive/MyDrive/298/Elec/NovTest/results/nov_fgsm_prediction_diversity.csv


In [ ]:
# transferability matrix

xfer_rows_nov = []
for eps in RFGSM_EVAL_EPS_LIST:
    alpha = eps * RFGSM_ALPHA_FACTOR
    print(f"\n[NovEval] Computing NOV transfer matrix for eps={eps}...")
    for i, atk_name in enumerate(student_names_nov):
        for j, def_name in enumerate(student_names_nov):
            atk_model = adv_students_nov[i]
            def_model = adv_students_nov[j]
            rmse_ij = eval_pair_rfgsm(def_model, atk_model, test_dl, eps=eps, alpha=alpha)
            xfer_rows_nov.append({
                "epsilon": eps,
                "attacker": atk_name,
                "defender": def_name,
                "rmse_adv": rmse_ij,
            })
            print(f"  atk={atk_name} -> def={def_name} | eps={eps:.2f} | RMSE={rmse_ij:.5f}")

df_xfer_nov = pd.DataFrame(xfer_rows_nov)
df_xfer_nov.to_csv(NOV_XFER_MATRIX_CSV, index=False)
print("\n[NovEval] Saved NOV transfer matrix to:", NOV_XFER_MATRIX_CSV)


[NovEval] Computing NOV transfer matrix for eps=0.1...
  atk=student_05_enc_attn_rfgsm_adv -> def=student_05_enc_attn_rfgsm_adv | eps=0.10 | RMSE=0.14281
  atk=student_05_enc_attn_rfgsm_adv -> def=student_06_dec_attn_rfgsm_adv | eps=0.10 | RMSE=0.14597
  atk=student_05_enc_attn_rfgsm_adv -> def=student_07_ffn_rfgsm_adv | eps=0.10 | RMSE=0.13535
  atk=student_05_enc_attn_rfgsm_adv -> def=student_08_norms_rfgsm_adv | eps=0.10 | RMSE=0.14486
  atk=student_06_dec_attn_rfgsm_adv -> def=student_05_enc_attn_rfgsm_adv | eps=0.10 | RMSE=0.13903
  atk=student_06_dec_attn_rfgsm_adv -> def=student_06_dec_attn_rfgsm_adv | eps=0.10 | RMSE=0.15295
  atk=student_06_dec_attn_rfgsm_adv -> def=student_07_ffn_rfgsm_adv | eps=0.10 | RMSE=0.13634
  atk=student_06_dec_attn_rfgsm_adv -> def=student_08_norms_rfgsm_adv | eps=0.10 | RMSE=0.14837
  atk=student_07_ffn_rfgsm_adv -> def=student_05_enc_attn_rfgsm_adv | eps=0.10 | RMSE=0.13704
  atk=student_07_ffn_rfgsm_adv -> def=student_06_dec_attn_rfgsm_adv | eps=

In [ ]:
#single run eval

print("\n[NovEval] Single run eval: base vs NOV ensemble")
print("  Epsilons:", RFGSM_EVAL_EPS_LIST)

single_seed = 1234
random.seed(single_seed)
np.random.seed(single_seed)
torch.manual_seed(single_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(single_seed)

single_rows = []   # ["attack", "epsilon", "model", "RMSE"]

base_clean_single   = eval_clean_rmse(best_base,  test_dl)
nov_ens_clean_single  = eval_ensemble_rmse(adv_students_nov, test_dl)

single_rows.append(["clean", None, "base",           base_clean_single])
single_rows.append(["clean", None, "nov_ensemble",   nov_ens_clean_single])

print(
    f"[NovEval Single-run] clean | base={base_clean_single:.5f} | "
    f"nov_ensemble={nov_ens_clean_single:.5f}",
    flush=True
)

for eps in RFGSM_EVAL_EPS_LIST:
    alpha = eps * RFGSM_ALPHA_FACTOR

    base_adv_single = eval_pair_rfgsm(
        defender=best_base,
        attacker=best_base,
        loader=test_dl,
        eps=eps,
        alpha=alpha
    )

    nov_rand_single = eval_random_switch_rfgsm(
        models=adv_students_nov,
        loader=test_dl,
        eps=eps,
        alpha=alpha
    )

    single_rows.append(["rfgsm", eps, "base",        base_adv_single])
    single_rows.append(["rfgsm", eps, "nov_rand",    nov_rand_single])

    print(
        f"[NovEval Single-run] eps={eps:.2f} | base_adv={base_adv_single:.5f} | "
        f"nov_rand={nov_rand_single:.5f}",
        flush=True
    )

df_single_nov = pd.DataFrame(
    single_rows,
    columns=["attack", "epsilon", "model", "RMSE"]
)
df_single_nov.to_csv(NOV_SINGLE_RUN_CSV, index=False)
print("\n[NovEval] Saved NOV single-run eval CSV to:", NOV_SINGLE_RUN_CSV)


[NovEval] Single run eval: base vs NOV ensemble
  Epsilons: [0.1, 0.2, 0.3, 0.5]
[NovEval Single-run] clean | base=0.11506 | nov_ensemble=0.11770
[NovEval Single-run] eps=0.10 | base_adv=0.16904 | nov_rand=0.14352
[NovEval Single-run] eps=0.20 | base_adv=0.23030 | nov_rand=0.13856
[NovEval Single-run] eps=0.30 | base_adv=0.27071 | nov_rand=0.14917
[NovEval Single-run] eps=0.50 | base_adv=0.29807 | nov_rand=0.15060

[NovEval] Saved NOV single-run eval CSV to: /content/drive/MyDrive/298/Elec/NovTest/results/nov_fgsm_morphence_single_run.csv


In [ ]:
# 30 run eval

N_RUNS_LARGE = 30

NOV_FGSM_RUNS_30_CSV  = os.path.join(RESULTS_DIR, "nov_fgsm_morphence_runs_30.csv")
NOV_FGSM_STATS_30_CSV = os.path.join(RESULTS_DIR, "nov_fgsm_morphence_stats_30.csv")

print("\n[NovEval] 30-run FGSM eval: base vs NOV ensemble")
print("  Epsilons:", RFGSM_EVAL_EPS_LIST)
print("  Runs:", N_RUNS_LARGE)

all_rows_nov_30 = []

start_all_30 = time.time()
hb_flag_30, hb_thread_30 = start_heartbeat("eval_nov_fgsm_morphence_30runs")

try:
    for run_i in range(1, N_RUNS_LARGE + 1):
        run_start = time.time()
        seed_i = 6000 + run_i
        random.seed(seed_i)
        np.random.seed(seed_i)
        torch.manual_seed(seed_i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed_i)

        base_clean = eval_clean_rmse(best_base, test_dl)
        nov_clean  = eval_ensemble_rmse(adv_students_nov, test_dl)

        all_rows_nov_30.append(["clean", None, "base",         base_clean,  run_i])
        all_rows_nov_30.append(["clean", None, "nov_ensemble", nov_clean,   run_i])

        print(
            f"[NovEval Run {run_i:02d}] clean | "
            f"base={base_clean:.5f} | nov_ens={nov_clean:.5f}",
            flush=True
        )


        for eps in RFGSM_EVAL_EPS_LIST:
            alpha = eps * RFGSM_ALPHA_FACTOR

            base_adv = eval_pair_rfgsm(
                defender=best_base,
                attacker=best_base,
                loader=test_dl,
                eps=eps,
                alpha=alpha
            )
            nov_rand = eval_random_switch_rfgsm(
                models=adv_students_nov,
                loader=test_dl,
                eps=eps,
                alpha=alpha
            )

            all_rows_nov_30.append(["rfgsm", eps, "base",     base_adv, run_i])
            all_rows_nov_30.append(["rfgsm", eps, "nov_rand", nov_rand, run_i])

            print(
                f"[NovEval Run {run_i:02d}] eps={eps:.2f} | "
                f"base_adv={base_adv:.5f} | nov_rand={nov_rand:.5f}",
                flush=True
            )

        df_runs_nov_30 = pd.DataFrame(
            all_rows_nov_30,
            columns=["attack", "epsilon", "model", "RMSE", "run"]
        )
        df_runs_nov_30.to_csv(NOV_FGSM_RUNS_30_CSV, index=False)

        stats_nov_30 = (
            df_runs_nov_30
            .groupby(["attack", "epsilon", "model"])["RMSE"]
            .agg(mean="mean", std="std", n="count")
            .reset_index()
            .sort_values(["attack", "epsilon", "model"])
        )
        stats_nov_30.to_csv(NOV_FGSM_STATS_30_CSV, index=False)

        run_mins   = (time.time() - run_start) / 60.0
        total_mins = (time.time() - start_all_30) / 60.0
        print(
            f"[NovEval] Completed run {run_i}/{N_RUNS_LARGE} | "
            f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
            flush=True
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()

finally:
    stop_heartbeat(hb_flag_30, hb_thread_30)

print("\n[NovEval] Saved 30 run NOV FGSM eval runs CSV to:", NOV_FGSM_RUNS_30_CSV)
print("[NovEval] Saved 30 run NOV FGSM eval stats CSV to:", NOV_FGSM_STATS_30_CSV)


[NovEval] 30-run FGSM eval: base vs NOV ensemble
  Epsilons: [0.1, 0.2, 0.3, 0.5]
  Runs: 30
[NovEval Run 01] clean | base=0.11506 | nov_ens=0.11770
[NovEval Run 01] eps=0.10 | base_adv=0.16954 | nov_rand=0.14320
[NovEval Run 01] eps=0.20 | base_adv=0.22987 | nov_rand=0.14028
[NovEval Run 01] eps=0.30 | base_adv=0.27251 | nov_rand=0.14799
[NovEval Run 01] eps=0.50 | base_adv=0.29940 | nov_rand=0.15239
[NovEval] Completed run 1/30 | run_time=1.41 min | total_elapsed=1.41 min
[NovEval Run 02] clean | base=0.11506 | nov_ens=0.11770
[NovEval Run 02] eps=0.10 | base_adv=0.16912 | nov_rand=0.14159
[NovEval Run 02] eps=0.20 | base_adv=0.23068 | nov_rand=0.13891
[NovEval Run 02] eps=0.30 | base_adv=0.27103 | nov_rand=0.14760
[NovEval Run 02] eps=0.50 | base_adv=0.29750 | nov_rand=0.14999
[NovEval] Completed run 2/30 | run_time=1.39 min | total_elapsed=2.81 min
[NovEval Run 03] clean | base=0.11506 | nov_ens=0.11770
[NovEval Run 03] eps=0.10 | base_adv=0.16959 | nov_rand=0.14476
[NovEval Run 0

In [ ]:
# FGSM stats

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
NOV_PROJECT_DIR = "/content/drive/MyDrive/298/Elec/NovTest"
RESULTS_DIR     = os.path.join(NOV_PROJECT_DIR, "results")
FIG_DIR         = RESULTS_DIR

NOV_FGSM_RUNS_30_CSV  = os.path.join(RESULTS_DIR, "nov_fgsm_morphence_runs_30.csv")

print("RESULTS_DIR:", RESULTS_DIR)
print("NOV_FGSM_RUNS_30_CSV:", NOV_FGSM_RUNS_30_CSV)
os.makedirs(FIG_DIR, exist_ok=True)

RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/results
NOV_FGSM_RUNS_30_CSV: /content/drive/MyDrive/298/Elec/NovTest/results/nov_fgsm_morphence_runs_30.csv


In [ ]:
df_runs = pd.read_csv(NOV_FGSM_RUNS_30_CSV)
print("Loaded NOV 30 run runs:", df_runs.shape)
print(df_runs.head())

Loaded NOV 30 run runs: (300, 5)
  attack  epsilon         model      RMSE  run
0  clean      NaN          base  0.115062    1
1  clean      NaN  nov_ensemble  0.117705    1
2  rfgsm      0.1          base  0.169537    1
3  rfgsm      0.1      nov_rand  0.143201    1
4  rfgsm      0.2          base  0.229872    1


In [ ]:
df_clean = df_runs[df_runs["attack"] == "clean"].copy()

# enforce ordering
# base first, then nov_ensemble
df_clean["model"] = pd.Categorical(
    df_clean["model"],
    categories=["base", "nov_ensemble"],
    ordered=True
)

clean_stats_30 = (
    df_clean
    .groupby("model", observed=False)["RMSE"]
    .agg(
        mean   ="mean",
        std    ="std",
        min    ="min",
        q1     =lambda s: s.quantile(0.25),
        median ="median",
        q3     =lambda s: s.quantile(0.75),
        max    ="max",
        n      ="count",
    )
    .reset_index()
    .sort_values("model")
)

print("\n[NOV 30 run | Clean] stats:")
print(clean_stats_30.to_string(index=False))

clean_csv_30 = os.path.join(FIG_DIR, "nov_fgsm_clean_stats_30.csv")
clean_stats_30.to_csv(clean_csv_30, index=False)
print("Saved NOV 30 run clean stats to:", clean_csv_30)



df_rfgsm = df_runs[df_runs["attack"] == "rfgsm"].copy()

def model_rfgsm_stats_nov(df, model_name, prefix="nov_fgsm_30"):
    """
    Aggregate RMSE over 30 runs for one model ("base" or "nov_rand"),
    grouped by epsilon, and save to CSV.
    """
    st = (
        df[df["model"] == model_name]
        .groupby("epsilon", observed=False)["RMSE"]
        .agg(
            mean   ="mean",
            std    ="std",
            min    ="min",
            q1     =lambda s: s.quantile(0.25),
            median ="median",
            q3     =lambda s: s.quantile(0.75),
            max    ="max",
            n      ="count",
        )
        .reset_index()
        .sort_values("epsilon")
    )

    print(f"\n[NOV 30 run | rFGSM] {model_name} stats:")
    print(st.to_string(index=False))

    out_csv = os.path.join(FIG_DIR, f"{prefix}_rfgsm_{model_name}_stats.csv")
    st.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    return st

base_stats_30     = model_rfgsm_stats_nov(df_rfgsm, "base")
nov_rand_stats_30 = model_rfgsm_stats_nov(df_rfgsm, "nov_rand")


[NOV 30 run | Clean] stats:
       model     mean  std      min       q1   median       q3      max  n
        base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
nov_ensemble 0.117705  0.0 0.117705 0.117705 0.117705 0.117705 0.117705 30
Saved NOV 30 run clean stats to: /content/drive/MyDrive/298/Elec/NovTest/results/nov_fgsm_clean_stats_30.csv

[NOV 30 run | rFGSM] base stats:
 epsilon     mean      std      min       q1   median       q3      max  n
     0.1 0.169486 0.000443 0.168608 0.169197 0.169478 0.169801 0.170392 30
     0.2 0.230340 0.000729 0.229203 0.229933 0.230224 0.230780 0.232327 30
     0.3 0.272287 0.000682 0.270616 0.271891 0.272195 0.272900 0.273290 30
     0.5 0.298049 0.000667 0.296412 0.297790 0.298019 0.298479 0.299402 30
Saved: /content/drive/MyDrive/298/Elec/NovTest/results/nov_fgsm_30_rfgsm_base_stats.csv

[NOV 30 run | rFGSM] nov_rand stats:
 epsilon     mean      std      min       q1   median       q3      max  n
     0.1 0.142991 0.000970 

In [ ]:
FINAL_RESULTS_DIR = "/content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results"
NOV_RESULTS_DIR   = "/content/drive/MyDrive/298/Elec/NovTest/results"

FGSM_RUNS_CSV          = os.path.join(FINAL_RESULTS_DIR, "fgsm_morphence_runs.csv")
NOV_FGSM_RUNS_30_CSV   = os.path.join(NOV_RESULTS_DIR, "nov_fgsm_morphence_runs_30.csv")

print("FinalPipeline FGSM runs CSV:", FGSM_RUNS_CSV)
print("NOV 30-run FGSM runs CSV:", NOV_FGSM_RUNS_30_CSV)

FinalPipeline FGSM runs CSV: /content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results/fgsm_morphence_runs.csv
NOV 30-run FGSM runs CSV: /content/drive/MyDrive/298/Elec/NovTest/results/nov_fgsm_morphence_runs_30.csv


In [ ]:
df_orig = pd.read_csv(FGSM_RUNS_CSV)
df_nov  = pd.read_csv(NOV_FGSM_RUNS_30_CSV)

print("Loaded original FGSM runs:", df_orig.shape)
print(df_orig.head())
print("\nLoaded NOV 30 run FGSM runs:", df_nov.shape)
print(df_nov.head())

df_nov_no_base = df_nov[df_nov["model"] != "base"].copy()

COMBINED_DIR = NOV_RESULTS_DIR
os.makedirs(COMBINED_DIR, exist_ok=True)

Loaded original FGSM runs: (300, 5)
  attack  epsilon           model      RMSE  run
0  clean      NaN            base  0.115062    1
1  clean      NaN  morph_ensemble  0.117630    1
2  rfgsm      0.1            base  0.169737    1
3  rfgsm      0.1      morph_rand  0.142028    1
4  rfgsm      0.2            base  0.229181    1

Loaded NOV 30 run FGSM runs: (300, 5)
  attack  epsilon         model      RMSE  run
0  clean      NaN          base  0.115062    1
1  clean      NaN  nov_ensemble  0.117705    1
2  rfgsm      0.1          base  0.169537    1
3  rfgsm      0.1      nov_rand  0.143201    1
4  rfgsm      0.2          base  0.229872    1


In [ ]:
df_clean_orig = df_orig[df_orig["attack"] == "clean"].copy()
df_clean_nov  = df_nov_no_base[df_nov_no_base["attack"] == "clean"].copy()

df_clean_orig = df_clean_orig[df_clean_orig["model"].isin(["base", "morph_ensemble"])]
df_clean_nov  = df_clean_nov[df_clean_nov["model"].isin(["nov_ensemble"])]

df_clean_comb = pd.concat([df_clean_orig, df_clean_nov], ignore_index=True)

df_clean_comb["model"] = pd.Categorical(
    df_clean_comb["model"],
    categories=["base", "morph_ensemble", "nov_ensemble"],
    ordered=True
)

clean_comp_stats_30 = (
    df_clean_comb
    .groupby("model", observed=False)["RMSE"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        q1=lambda s: s.quantile(0.25),
        median="median",
        q3=lambda s: s.quantile(0.75),
        max="max",
        n="count",
    )
    .reset_index()
    .sort_values("model")
)

print("\n[COMPARATIVE CLEAN | 30 runs] base vs morph_ensemble vs nov_ensemble:")
print(clean_comp_stats_30.to_string(index=False))

clean_comp_csv_30 = os.path.join(COMBINED_DIR, "fgsm_clean_stats_orig_vs_nov_30runs.csv")
clean_comp_stats_30.to_csv(clean_comp_csv_30, index=False)
print("Saved comparative 30 run clean stats to:", clean_comp_csv_30)

# FGSM STATS
# base vs morph_rand vs nov_rand (per epsilon)
df_rfgsm_orig = df_orig[df_orig["attack"] == "rfgsm"].copy()
df_rfgsm_nov  = df_nov_no_base[df_nov_no_base["attack"] == "rfgsm"].copy()

df_rfgsm_orig = df_rfgsm_orig[df_rfgsm_orig["model"].isin(["base", "morph_rand"])]
df_rfgsm_nov  = df_rfgsm_nov[df_rfgsm_nov["model"].isin(["nov_rand"])]

df_rfgsm_comb = pd.concat([df_rfgsm_orig, df_rfgsm_nov], ignore_index=True)

df_rfgsm_comb["model"] = pd.Categorical(
    df_rfgsm_comb["model"],
    categories=["base", "morph_rand", "nov_rand"],
    ordered=True
)

rfgsm_comp_stats_30 = (
    df_rfgsm_comb
    .groupby(["epsilon", "model"], observed=False)["RMSE"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        q1=lambda s: s.quantile(0.25),
        median="median",
        q3=lambda s: s.quantile(0.75),
        max="max",
        n="count",
    )
    .reset_index()
    .sort_values(["epsilon", "model"])
)

print("\n[COMPARATIVE rFGSM | 30 runs] base vs morph_rand vs nov_rand:")
print(rfgsm_comp_stats_30.to_string(index=False))

rfgsm_comp_csv_30 = os.path.join(COMBINED_DIR, "fgsm_rfgsm_stats_orig_vs_nov_30runs.csv")
rfgsm_comp_stats_30.to_csv(rfgsm_comp_csv_30, index=False)
print("Saved comparative 30-run rFGSM stats to:", rfgsm_comp_csv_30)


[COMPARATIVE CLEAN | 30 runs] base vs morph_ensemble vs nov_ensemble:
         model     mean  std      min       q1   median       q3      max  n
          base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
morph_ensemble 0.117630  0.0 0.117630 0.117630 0.117630 0.117630 0.117630 30
  nov_ensemble 0.117705  0.0 0.117705 0.117705 0.117705 0.117705 0.117705 30
Saved comparative 30 run clean stats to: /content/drive/MyDrive/298/Elec/NovTest/results/fgsm_clean_stats_orig_vs_nov_30runs.csv

[COMPARATIVE rFGSM | 30 runs] base vs morph_rand vs nov_rand:
 epsilon      model     mean      std      min       q1   median       q3      max  n
     0.1       base 0.169164 0.000460 0.168365 0.168848 0.169102 0.169434 0.170091 30
     0.1 morph_rand 0.141715 0.000960 0.139935 0.141192 0.141727 0.142082 0.144402 30
     0.1   nov_rand 0.142991 0.000970 0.141586 0.142062 0.143177 0.143623 0.144758 30
     0.2       base 0.230447 0.000853 0.228434 0.229911 0.230401 0.230958 0.232460 30

In [ ]:
#BIM

In [ ]:
import traceback

In [ ]:
NOV_BIM_PROJECT_DIR    = "/content/drive/MyDrive/298/Elec/NovTest/BIM"

BIM_STUDENTS_ADV_DIR   = os.path.join(NOV_BIM_PROJECT_DIR, "students_adv_nov_bim")
BIM_RESULTS_DIR        = os.path.join(NOV_BIM_PROJECT_DIR, "results")
BIM_PLOTS_DIR          = os.path.join(BIM_RESULTS_DIR, "plots")
BIM_ARTIFACTS_DIR      = os.path.join(BIM_RESULTS_DIR, "artifacts")
BIM_CRASH_DIR          = os.path.join(NOV_BIM_PROJECT_DIR, "crash_dump")

BIM_RUNS_CSV           = os.path.join(BIM_RESULTS_DIR, "bim_nov_runs.csv")
BIM_STATS_CSV          = os.path.join(BIM_RESULTS_DIR, "bim_nov_stats.csv")
BIM_PER_HORIZON_CSV    = os.path.join(BIM_RESULTS_DIR, "bim_nov_per_horizon_rmse.csv")
BIM_GRAD_NORMS_CSV     = os.path.join(BIM_RESULTS_DIR, "bim_nov_grad_norms.csv")
BIM_MODEL_SIZES_CSV    = os.path.join(BIM_RESULTS_DIR, "bim_nov_model_sizes.csv")
BIM_ARTIFACT_INDEX_CSV = os.path.join(BIM_RESULTS_DIR, "bim_nov_artifact_index.csv")
BIM_ERROR_LOG          = os.path.join(BIM_CRASH_DIR, "error_log.txt")

for d in [
    NOV_BIM_PROJECT_DIR,
    BIM_STUDENTS_ADV_DIR,
    BIM_RESULTS_DIR,
    BIM_PLOTS_DIR,
    BIM_ARTIFACTS_DIR,
    BIM_CRASH_DIR,
]:
    os.makedirs(d, exist_ok=True)

print("\n[BIM-Nov] NOV_BIM_PROJECT_DIR:", NOV_BIM_PROJECT_DIR)
print("[BIM-Nov] BIM_STUDENTS_ADV_DIR:", BIM_STUDENTS_ADV_DIR)
print("[BIM-Nov] BIM_RESULTS_DIR:", BIM_RESULTS_DIR)


[BIM-Nov] NOV_BIM_PROJECT_DIR: /content/drive/MyDrive/298/Elec/NovTest/BIM
[BIM-Nov] BIM_STUDENTS_ADV_DIR: /content/drive/MyDrive/298/Elec/NovTest/BIM/students_adv_nov_bim
[BIM-Nov] BIM_RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/BIM/results


In [ ]:
#bim hyperparams

BIM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
BIM_EVAL_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]
BIM_ITERS          = 10

BIM_RANDOM_START_TRAIN = False
BIM_RANDOM_START_EVAL  = True

def log_error_bim(msg: str):
    with open(BIM_ERROR_LOG, "a") as f:
        f.write(msg + "\n")

In [ ]:
#bim attack

def bim_attack(attacker_model,
               X,
               Y,
               eps,
               alpha,
               iters,
               random_start=False,
               return_grad_norm=False):
    """
    Basic Iterative Method (BIM) attack.

    attacker_model : nn.Module used to generate adversarial examples
    X, Y           : clean inputs/targets
    eps            : L_infinity budget
    alpha          : step size
    iters          : number of gradient steps
    random_start   : if True, start from a random point in [-eps, eps]
    return_grad_norm : if True, also returns final gradient L2 norms
    """
    attacker_model.eval()

    if random_start:
        noise = torch.empty_like(X).uniform_(-eps, eps)
        X_adv = (X + noise).clamp(0.0, 1.0)
    else:
        X_adv = X.clone()

    X_adv = X_adv.detach().requires_grad_(True)
    last_grad = None

    for _ in range(iters):
        preds = attacker_model(X_adv)
        loss  = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad = X_adv.grad
        last_grad = grad

        X_adv = (X_adv + alpha * grad.sign()).detach()
        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach().requires_grad_(True)

    X_adv = X_adv.detach()
    if return_grad_norm and last_grad is not None:
        gn = last_grad.view(last_grad.size(0), -1).norm(p=2, dim=1)
        return X_adv, gn.detach()
    return X_adv


In [ ]:
# reconstruct novel_student_paths from existing files

STUDENTS_DIR = "/content/drive/MyDrive/298/Elec/NovTest/students_nov"

print("Scanning for novel student seeds in:", STUDENTS_DIR)

novel_student_paths = []

expected_map = {
    "student_05_enc_attn.pth": 5,
    "student_06_dec_attn.pth": 6,
    "student_07_ffn.pth": 7,
    "student_08_norms.pth": 8,
}

for fname in os.listdir(STUDENTS_DIR):
    if fname in expected_map:
        idx = expected_map[fname]
        name = fname.replace(".pth", "")
        full_path = os.path.join(STUDENTS_DIR, fname)
        novel_student_paths.append((idx, name, full_path))
        print(f"  Found seed student {idx}: {name} -> {full_path}")

# sort by student index (5,6,7,8)
novel_student_paths = sorted(novel_student_paths, key=lambda x: x[0])

print("\nLoaded novel_student_paths:")
for t in novel_student_paths:
    print(" ", t)

# safety check
assert len(novel_student_paths) == 4, \
    f"Expected 4 novel students, found {len(novel_student_paths)}"

Scanning for novel student seeds in: /content/drive/MyDrive/298/Elec/NovTest/students_nov
  Found seed student 5: student_05_enc_attn -> /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_05_enc_attn.pth
  Found seed student 6: student_06_dec_attn -> /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_06_dec_attn.pth
  Found seed student 7: student_07_ffn -> /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_07_ffn.pth
  Found seed student 8: student_08_norms -> /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_08_norms.pth

Loaded novel_student_paths:
  (5, 'student_05_enc_attn', '/content/drive/MyDrive/298/Elec/NovTest/students_nov/student_05_enc_attn.pth')
  (6, 'student_06_dec_attn', '/content/drive/MyDrive/298/Elec/NovTest/students_nov/student_06_dec_attn.pth')
  (7, 'student_07_ffn', '/content/drive/MyDrive/298/Elec/NovTest/students_nov/student_07_ffn.pth')
  (8, 'student_08_norms', '/content/drive/MyDrive/298/Elec/NovTest/students_nov/stud

In [ ]:
novel_bim_adv_paths = []

print("\n[BIM-Nov] BIM adversarial training of novel students")

# safety check
# ensure novel_student_paths (untrained seeds) exists
assert "novel_student_paths" in globals(), \
    "[BIM-Nov] novel_student_paths not found. Run NOV student generation first."

for idx, name, sp in novel_student_paths:
    # name is like "student_05_enc_attn", sp is the ORIGINAL noisy seed .pth
    print(f"\n[BIM-Nov] [{name} | idx={idx}] Starting BIM adversarial training...")
    print(f"  Loading NOV seed from: {sp}")

    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))

    opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)

    stu_history = []

    hb_flag, hb_thread = start_heartbeat(f"{name}_bim_train")
    start_time = time.time()

    try:
        for ep in range(1, EPOCHS_STU + 1):
            stu.train()
            tot = cnt = 0

            for xb, yb in train_dl:
                xb = xb.to(device)
                yb = yb.to(device)

                # clean prediction loss
                pred_c = stu(xb)
                Lc = rmse_only(pred_c, yb)

                # multi-epsilon BIM adversarial loss
                adv_losses = []
                for eps in BIM_TRAIN_EPS_LIST:
                    alpha = eps / float(BIM_ITERS)
                    xa = bim_attack(
                        attacker_model=stu,
                        X=xb,
                        Y=yb,
                        eps=eps,
                        alpha=alpha,
                        iters=BIM_ITERS,
                        random_start=BIM_RANDOM_START_TRAIN,
                        return_grad_norm=False
                    )
                    pred_a = stu(xa)
                    La = rmse_only(pred_a, yb)
                    adv_losses.append(La)

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                # robust loss mix (same 0.25 / 0.75 as FGSM pipeline)
                loss = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot += loss.item() * bs
                cnt += bs

            avg_loss = tot / max(1, cnt)
            elapsed  = (time.time() - start_time) / 60.0
            print(
                f"  [BIM-Nov {name}] Epoch {ep:02d} | "
                f"avg_loss={avg_loss:.4f} | elapsed={elapsed:.2f} min",
                flush=True
            )

            stu_history.append({
                "student_idx": int(idx),
                "student_name": name,
                "epoch": ep,
                "avg_loss": avg_loss,
                "elapsed_min": elapsed,
            })

    except Exception as e:
        # on crash, stop heartbeat, log error, and save partial checkpoint
        stop_heartbeat(hb_flag, hb_thread)
        msg = (
            f"[BIM-Nov {name}] BIM adv training crash: {repr(e)}\n"
            f"{traceback.format_exc()}"
        )
        print(msg)
        log_error_bim(msg)
        ckpt_crash = os.path.join(BIM_STUDENTS_ADV_DIR, f"{name}_bim_CRASHED.pth")
        torch.save(stu.to("cpu").state_dict(), ckpt_crash)
        raise

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    # save per-student BIM training history CSV
    stu_hist_path = os.path.join(BIM_RESULTS_DIR, f"{name}_bim_train_history.csv")
    df_stu_hist = pd.DataFrame(stu_history)
    df_stu_hist.to_csv(stu_hist_path, index=False)
    print(f"  [BIM-Nov {name}] Saved BIM training history to:", stu_hist_path)

    # save BIM-adv student checkpoint
    outp_bim = os.path.join(BIM_STUDENTS_ADV_DIR, f"{name}_bim_adv.pth")
    torch.save(stu.to("cpu").state_dict(), outp_bim)
    novel_bim_adv_paths.append((idx, name, outp_bim))
    print(f"  [BIM-Nov {name}] Saved BIM adv student:", outp_bim)

    # log BIM-adv student model size
    size_mb = model_file_size_mb(outp_bim)
    df_ms = pd.DataFrame([{
        "model_name": f"{name}_bim_adv",
        "path": outp_bim,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, BIM_MODEL_SIZES_CSV)

novel_bim_adv_paths = sorted(novel_bim_adv_paths, key=lambda x: x[0])

print("\n[BIM-Nov] BIM adv trained NOV students:")
for idx, name, path in novel_bim_adv_paths:
    print(f"  idx={idx} | {name}_bim_adv - {path}")


[BIM-Nov] BIM adversarial training of novel students

[BIM-Nov] [student_05_enc_attn | idx=5] Starting BIM adversarial training...
  Loading NOV seed from: /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_05_enc_attn.pth


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[heartbeat:student_05_enc_attn_bim_train] running... elapsed=10.00 min
[heartbeat:student_05_enc_attn_bim_train] running... elapsed=20.00 min
  [BIM-Nov student_05_enc_attn] Epoch 01 | avg_loss=0.1447 | elapsed=23.88 min
[heartbeat:student_05_enc_attn_bim_train] running... elapsed=30.00 min
[heartbeat:student_05_enc_attn_bim_train] running... elapsed=40.00 min
  [BIM-Nov student_05_enc_attn] Epoch 02 | avg_loss=0.1420 | elapsed=47.89 min
[heartbeat:student_05_enc_attn_bim_train] running... elapsed=50.00 min
[heartbeat:student_05_enc_attn_bim_train] running... elapsed=60.00 min
[heartbeat:student_05_enc_attn_bim_train] running... elapsed=70.00 min
  [BIM-Nov student_05_enc_attn] Epoch 03 | avg_loss=0.1371 | elapsed=71.83 min
[heartbeat:student_05_enc_attn_bim_train] running... elapsed=80.00 min
[heartbeat:student_05_enc_attn_bim_train] running... elapsed=90.00 min
  [BIM-Nov student_05_enc_attn] Epoch 04 | avg_loss=0.1334 | elapsed=95.82 min
[heartbeat:student_05_enc_attn_bim_train] run

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[heartbeat:student_06_dec_attn_bim_train] running... elapsed=10.00 min
[heartbeat:student_06_dec_attn_bim_train] running... elapsed=20.00 min
  [BIM-Nov student_06_dec_attn] Epoch 01 | avg_loss=0.1448 | elapsed=24.33 min
[heartbeat:student_06_dec_attn_bim_train] running... elapsed=30.00 min
[heartbeat:student_06_dec_attn_bim_train] running... elapsed=40.00 min
  [BIM-Nov student_06_dec_attn] Epoch 02 | avg_loss=0.1416 | elapsed=48.74 min
[heartbeat:student_06_dec_attn_bim_train] running... elapsed=50.00 min
[heartbeat:student_06_dec_attn_bim_train] running... elapsed=60.00 min
[heartbeat:student_06_dec_attn_bim_train] running... elapsed=70.00 min
  [BIM-Nov student_06_dec_attn] Epoch 03 | avg_loss=0.1363 | elapsed=73.17 min
[heartbeat:student_06_dec_attn_bim_train] running... elapsed=80.00 min
[heartbeat:student_06_dec_attn_bim_train] running... elapsed=90.00 min
  [BIM-Nov student_06_dec_attn] Epoch 04 | avg_loss=0.1326 | elapsed=97.57 min
[heartbeat:student_06_dec_attn_bim_train] run

In [ ]:
import os
import numpy as np
import pandas as pd
import torch

In [ ]:
BIM_NOV_WEIGHT_DIVERSITY_CSV = os.path.join(BIM_RESULTS_DIR, "bim_nov_weight_diversity.csv")
BIM_NOV_PRED_DIVERSITY_CSV   = os.path.join(BIM_RESULTS_DIR, "bim_nov_prediction_diversity.csv")

In [ ]:
print("\n[NOV-BIM] Loading BIM adversarially-trained novel students...")

nov_bim_paths = sorted(
    [
        os.path.join(BIM_STUDENTS_ADV_DIR, f)
        for f in os.listdir(BIM_STUDENTS_ADV_DIR)
        if f.endswith("_bim_adv.pth")
    ]
)

assert len(nov_bim_paths) == 4, \
    f"Expected 4 BIM adv students (5–8). Found {len(nov_bim_paths)}."

nov_bim_students = []
nov_bim_names    = []

for p in nov_bim_paths:
    name = os.path.basename(p).replace(".pth", "")
    print("  Loading:", name)

    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)

    stu.load_state_dict(torch.load(p, map_location=device))
    stu.eval()

    nov_bim_students.append(stu)
    nov_bim_names.append(name)

print("\n[NOV-BIM] Loaded BIM novel students:")
for n in nov_bim_names:
    print(" ", n)


[NOV-BIM] Loading BIM adversarially-trained novel students...
  Loading: student_05_enc_attn_bim_adv


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  Loading: student_06_dec_attn_bim_adv
  Loading: student_07_ffn_bim_adv
  Loading: student_08_norms_bim_adv

[NOV-BIM] Loaded BIM novel students:
  student_05_enc_attn_bim_adv
  student_06_dec_attn_bim_adv
  student_07_ffn_bim_adv
  student_08_norms_bim_adv


In [ ]:
def flatten_params(model):
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])

def compute_weight_diversity_bim(students, names, out_csv=BIM_NOV_WEIGHT_DIVERSITY_CSV):
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)
    rows = []
    for i in range(k):
        for j in range(i + 1, k):
            d = torch.norm(vecs[i] - vecs[j], p=2).item()
            rows.append({
                "student_i": names[i],
                "student_j": names[j],
                "l2_distance": d,
            })
    df_w = pd.DataFrame(rows)
    df_w.to_csv(out_csv, index=False)
    print("[NOV-BIM] Saved weight diversity to:", out_csv)
    return df_w

@torch.no_grad()
def compute_prediction_diversity_bim(students, loader, out_csv=BIM_NOV_PRED_DIVERSITY_CSV):
    for m in students:
        m.eval()

    k = len(students)
    H = HZN_STEPS

    sum_pred  = np.zeros(H, dtype=np.float64)
    sum_pred2 = np.zeros(H, dtype=np.float64)
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device)
        preds_stu = []
        for m in students:
            preds_stu.append(m(xb).cpu().numpy())  # (B,H,1)
        preds_stu = np.stack(preds_stu, axis=0)     # (k,B,H,1)

        # mean prediction per student over batch, then average over students
        preds_mean_batch = preds_stu.mean(axis=1).squeeze(-1)  # (k,H)

        sum_pred  += preds_mean_batch.mean(axis=0)
        sum_pred2 += (preds_mean_batch ** 2).mean(axis=0)
        count     += 1

    mean_pred  = sum_pred / max(1, count)
    mean_pred2 = sum_pred2 / max(1, count)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "horizon_idx": np.arange(H),
        "mean_pred": mean_pred,
        "var_pred": var_pred,
    })
    df_p.to_csv(out_csv, index=False)
    print("[NOV-BIM] Saved prediction diversity to:", out_csv)
    return df_p

print("\n[NOV-BIM] Computing diversity metrics for BIM-trained novel students...")
_ = compute_weight_diversity_bim(nov_bim_students, nov_bim_names)
_ = compute_prediction_diversity_bim(nov_bim_students, test_dl)


[NOV-BIM] Computing diversity metrics for BIM-trained novel students...
[NOV-BIM] Saved weight diversity to: /content/drive/MyDrive/298/Elec/NovTest/BIM/results/bim_nov_weight_diversity.csv
[NOV-BIM] Saved prediction diversity to: /content/drive/MyDrive/298/Elec/NovTest/BIM/results/bim_nov_prediction_diversity.csv


In [ ]:
def eval_pair_bim(defender,
                  attacker,
                  loader,
                  eps,
                  iters=BIM_ITERS,
                  random_start=BIM_RANDOM_START_EVAL):
    """
    Evaluate defender under BIM attack crafted by attacker.
    Uses gradients only for the attacker (to build BIM examples).
    Defender forward is done under no_grad.
    """
    defender.eval()
    attacker.eval()

    ys, ps = [], []
    alpha = eps / float(iters)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        # craft BIM adversarial inputs with the attacker (needs grads)
        xa = bim_attack(
            attacker_model=attacker,
            X=xb,
            Y=yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
            return_grad_norm=False,
        )

        # evaluate defender on the crafted adversarial examples (no grads needed)
        with torch.no_grad():
            p = defender(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
BIM_NOV_XFER_MATRIX_CSV = os.path.join(BIM_RESULTS_DIR, "bim_nov_transfer_matrix.csv")

xfer_rows_bim_nov = []

print("\n[NOV-BIM] Computing NOV-BIM transfer matrix (BIM attack)...")
for eps in BIM_EVAL_EPS_LIST:
    print(f"  eps={eps:.2f} ...")
    for i, atk_name in enumerate(nov_bim_names):
        for j, def_name in enumerate(nov_bim_names):
            atk_model = nov_bim_students[i]
            def_model = nov_bim_students[j]

            rmse_ij = eval_pair_bim(
                defender=def_model,
                attacker=atk_model,
                loader=test_dl,
                eps=eps,
                iters=BIM_ITERS,
                random_start=BIM_RANDOM_START_EVAL,
            )

            xfer_rows_bim_nov.append({
                "epsilon": eps,
                "attacker": atk_name,
                "defender": def_name,
                "rmse_adv": rmse_ij,
            })
            print(f"    atk={atk_name} -> def={def_name} | eps={eps:.2f} | RMSE={rmse_ij:.5f}")

df_xfer_bim_nov = pd.DataFrame(xfer_rows_bim_nov)
df_xfer_bim_nov.to_csv(BIM_NOV_XFER_MATRIX_CSV, index=False)
print("\n[NOV-BIM] Saved NOV-BIM transfer matrix to:", BIM_NOV_XFER_MATRIX_CSV)


[NOV-BIM] Computing NOV-BIM transfer matrix (BIM attack)...
  eps=0.10 ...
    atk=student_05_enc_attn_bim_adv -> def=student_05_enc_attn_bim_adv | eps=0.10 | RMSE=0.20409
    atk=student_05_enc_attn_bim_adv -> def=student_06_dec_attn_bim_adv | eps=0.10 | RMSE=0.19281
    atk=student_05_enc_attn_bim_adv -> def=student_07_ffn_bim_adv | eps=0.10 | RMSE=0.20493
    atk=student_05_enc_attn_bim_adv -> def=student_08_norms_bim_adv | eps=0.10 | RMSE=0.19974
    atk=student_06_dec_attn_bim_adv -> def=student_05_enc_attn_bim_adv | eps=0.10 | RMSE=0.20124
    atk=student_06_dec_attn_bim_adv -> def=student_06_dec_attn_bim_adv | eps=0.10 | RMSE=0.19589
    atk=student_06_dec_attn_bim_adv -> def=student_07_ffn_bim_adv | eps=0.10 | RMSE=0.20548
    atk=student_06_dec_attn_bim_adv -> def=student_08_norms_bim_adv | eps=0.10 | RMSE=0.19918
    atk=student_07_ffn_bim_adv -> def=student_05_enc_attn_bim_adv | eps=0.10 | RMSE=0.20349
    atk=student_07_ffn_bim_adv -> def=student_06_dec_attn_bim_adv | eps=

In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import torch

In [ ]:
BIM_NOV_RUNS_30_CSV  = os.path.join(BIM_RESULTS_DIR, "bim_nov_runs_30.csv")
BIM_NOV_STATS_30_CSV = os.path.join(BIM_RESULTS_DIR, "bim_nov_stats_30.csv")

print("\n[NOV-BIM Eval] BIM_RESULTS_DIR:", BIM_RESULTS_DIR)
print("[NOV-BIM Eval] BIM_NOV_RUNS_30_CSV:", BIM_NOV_RUNS_30_CSV)


[NOV-BIM Eval] BIM_RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/BIM/results
[NOV-BIM Eval] BIM_NOV_RUNS_30_CSV: /content/drive/MyDrive/298/Elec/NovTest/BIM/results/bim_nov_runs_30.csv


In [ ]:
best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(BASE_CKPT, map_location=device))
best_base.eval()
print("[NOV-BIM Eval] Loaded base model from:", BASE_CKPT)

[NOV-BIM Eval] Loaded base model from: /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth


In [ ]:
nov_bim_paths = sorted(
    [
        os.path.join(BIM_STUDENTS_ADV_DIR, f)
        for f in os.listdir(BIM_STUDENTS_ADV_DIR)
        if f.endswith("_bim_adv.pth")
    ]
)

assert len(nov_bim_paths) > 0, "[NOV-BIM Eval] No *_bim_adv.pth found in BIM_STUDENTS_ADV_DIR!"
print("[NOV-BIM Eval] NOV BIM students:")
for p in nov_bim_paths:
    print("  ", p)

nov_bim_students = []
nov_bim_names    = []
for sp in nov_bim_paths:
    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))
    stu.eval()
    nov_bim_students.append(stu)
    nov_bim_names.append(os.path.basename(sp).replace(".pth", ""))

print(f"[NOV-BIM Eval] Loaded {len(nov_bim_students)} NOV BIM students.")


[NOV-BIM Eval] NOV BIM students:
   /content/drive/MyDrive/298/Elec/NovTest/BIM/students_adv_nov_bim/student_05_enc_attn_bim_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/BIM/students_adv_nov_bim/student_06_dec_attn_bim_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/BIM/students_adv_nov_bim/student_07_ffn_bim_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/BIM/students_adv_nov_bim/student_08_norms_bim_adv.pth
[NOV-BIM Eval] Loaded 4 NOV BIM students.


In [ ]:
@torch.no_grad()
def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        p  = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

@torch.no_grad()
def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        preds = []
        for m in models:
            preds.append(m(xb))
        preds = torch.stack(preds, dim=0).mean(dim=0)  # (B,H,D)
        ys.append(yb.cpu().numpy())
        ps.append(preds.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_pair_bim(defender, attacker, loader, eps, alpha=None):
    defender.eval()
    attacker.eval()
    if alpha is None:
        alpha = eps / float(BIM_ITERS)

    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        xa = bim_attack(
            attacker_model=attacker,
            X=xb,
            Y=yb,
            eps=eps,
            alpha=alpha,
            iters=BIM_ITERS,
            random_start=BIM_RANDOM_START_EVAL,
            return_grad_norm=False,
        )

        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_random_switch_bim(models, loader, eps, alpha=None):
    if alpha is None:
        alpha = eps / float(BIM_ITERS)

    for m in models:
        m.eval()

    ys, ps = [], []
    n_stu = len(models)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        atk_idx  = random.randrange(n_stu)
        def_idx  = random.randrange(n_stu)
        attacker = models[atk_idx]
        defender = models[def_idx]

        xa = bim_attack(
            attacker_model=attacker,
            X=xb,
            Y=yb,
            eps=eps,
            alpha=alpha,
            iters=BIM_ITERS,
            random_start=BIM_RANDOM_START_EVAL,
            return_grad_norm=False,
        )

        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
N_RUNS_LARGE = 30

print("\n[NOV-BIM Eval] 30-run BIM eval: base vs NOV BIM ensemble")
print("  Epsilons:", BIM_EVAL_EPS_LIST)
print("  Runs:", N_RUNS_LARGE)

all_rows_bim_nov = []   # ["attack", "epsilon", "model", "RMSE", "run"]

start_all_30 = time.time()
hb_flag_30, hb_thread_30 = start_heartbeat("eval_nov_bim_30runs")

try:
    for run_i in range(1, N_RUNS_LARGE + 1):
        run_start = time.time()
        seed_i = 9000 + run_i
        random.seed(seed_i)
        np.random.seed(seed_i)
        torch.manual_seed(seed_i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed_i)

        # clean RMSE
        base_clean = eval_clean_rmse(best_base,       test_dl)
        nov_clean  = eval_ensemble_rmse(nov_bim_students, test_dl)

        all_rows_bim_nov.append(["clean", None, "base",             base_clean, run_i])
        all_rows_bim_nov.append(["clean", None, "nov_bim_ensemble", nov_clean,  run_i])

        print(
            f"[NOV-BIM Run {run_i:02d}] clean | "
            f"base={base_clean:.5f} | nov_bim_ens={nov_clean:.5f}",
            flush=True
        )

        # BIM attack per epsilon
        for eps in BIM_EVAL_EPS_LIST:
            alpha = eps / float(BIM_ITERS)

            base_adv = eval_pair_bim(
                defender=best_base,
                attacker=best_base,
                loader=test_dl,
                eps=eps,
                alpha=alpha,
            )

            nov_rand = eval_random_switch_bim(
                models=nov_bim_students,
                loader=test_dl,
                eps=eps,
                alpha=alpha,
            )

            all_rows_bim_nov.append(["bim", eps, "base",         base_adv, run_i])
            all_rows_bim_nov.append(["bim", eps, "nov_bim_rand", nov_rand, run_i])

            print(
                f"[NOV-BIM Run {run_i:02d}] eps={eps:.2f} | "
                f"base_adv={base_adv:.5f} | nov_bim_rand={nov_rand:.5f}",
                flush=True
            )

        # autosave after each run
        df_runs_bim_nov = pd.DataFrame(
            all_rows_bim_nov,
            columns=["attack", "epsilon", "model", "RMSE", "run"]
        )
        df_runs_bim_nov.to_csv(BIM_NOV_RUNS_30_CSV, index=False)

        stats_bim_nov = (
            df_runs_bim_nov
            .groupby(["attack", "epsilon", "model"])["RMSE"]
            .agg(mean="mean", std="std", n="count")
            .reset_index()
            .sort_values(["attack", "epsilon", "model"])
        )
        stats_bim_nov.to_csv(BIM_NOV_STATS_30_CSV, index=False)

        run_mins   = (time.time() - run_start) / 60.0
        total_mins = (time.time() - start_all_30) / 60.0
        print(
            f"[NOV-BIM Eval] Completed run {run_i}/{N_RUNS_LARGE} | "
            f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
            flush=True
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()

finally:
    stop_heartbeat(hb_flag_30, hb_thread_30)

print("\n[NOV-BIM Eval] Saved 30-run NOV BIM runs CSV to:", BIM_NOV_RUNS_30_CSV)
print("[NOV-BIM Eval] Saved 30-run NOV BIM stats CSV to:", BIM_NOV_STATS_30_CSV)


[NOV-BIM Eval] 30-run BIM eval: base vs NOV BIM ensemble
  Epsilons: [0.1, 0.2, 0.3, 0.5]
  Runs: 30
[NOV-BIM Run 01] clean | base=0.11506 | nov_bim_ens=0.17730
[NOV-BIM Run 01] eps=0.10 | base_adv=0.25027 | nov_bim_rand=0.20157
[NOV-BIM Run 01] eps=0.20 | base_adv=0.33719 | nov_bim_rand=0.20552
[NOV-BIM Run 01] eps=0.30 | base_adv=0.36850 | nov_bim_rand=0.21129
[heartbeat:eval_nov_bim_30runs] running... elapsed=10.00 min
[NOV-BIM Run 01] eps=0.50 | base_adv=0.39575 | nov_bim_rand=0.24581
[NOV-BIM Eval] Completed run 1/30 | run_time=11.50 min | total_elapsed=11.50 min
[NOV-BIM Run 02] clean | base=0.11506 | nov_bim_ens=0.17730
[NOV-BIM Run 02] eps=0.10 | base_adv=0.24898 | nov_bim_rand=0.20020
[NOV-BIM Run 02] eps=0.20 | base_adv=0.33940 | nov_bim_rand=0.20456
[heartbeat:eval_nov_bim_30runs] running... elapsed=20.00 min
[NOV-BIM Run 02] eps=0.30 | base_adv=0.36832 | nov_bim_rand=0.21039
[NOV-BIM Run 02] eps=0.50 | base_adv=0.39675 | nov_bim_rand=0.24546
[NOV-BIM Eval] Completed run 2/

In [ ]:
# BIM stats

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
BIM_FINAL_RESULTS_DIR = "/content/drive/MyDrive/298/Elec/FinalPipeline/bim/results"
BIM_NOV_RESULTS_DIR   = "/content/drive/MyDrive/298/Elec/NovTest/BIM/results"

BIM_RUNS_CSV_ORIG   = os.path.join(BIM_FINAL_RESULTS_DIR, "bim_morphence_runs.csv")
BIM_NOV_RUNS_30_CSV = os.path.join(BIM_NOV_RESULTS_DIR, "bim_nov_runs_30.csv")

print("Morphence BIM runs CSV:", BIM_RUNS_CSV_ORIG)
print("NOV BIM 30-run runs CSV:", BIM_NOV_RUNS_30_CSV)


Morphence BIM runs CSV: /content/drive/MyDrive/298/Elec/FinalPipeline/bim/results/bim_morphence_runs.csv
NOV BIM 30-run runs CSV: /content/drive/MyDrive/298/Elec/NovTest/BIM/results/bim_nov_runs_30.csv


In [ ]:
BIM_NOV_RESULTS_DIR = "/content/drive/MyDrive/298/Elec/NovTest/BIM/results"
BIM_NOV_RUNS_30_CSV = os.path.join(BIM_NOV_RESULTS_DIR, "bim_nov_runs_30.csv")
BIM_NOV_FIG_DIR     = BIM_NOV_RESULTS_DIR

print("BIM_NOV_RESULTS_DIR:", BIM_NOV_RESULTS_DIR)
print("BIM_NOV_RUNS_30_CSV:", BIM_NOV_RUNS_30_CSV)
os.makedirs(BIM_NOV_FIG_DIR, exist_ok=True)

df_nov_bim = pd.read_csv(BIM_NOV_RUNS_30_CSV)
print("Loaded NOV BIM 30-run runs:", df_nov_bim.shape)
print(df_nov_bim.head())

BIM_NOV_RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/BIM/results
BIM_NOV_RUNS_30_CSV: /content/drive/MyDrive/298/Elec/NovTest/BIM/results/bim_nov_runs_30.csv
Loaded NOV BIM 30-run runs: (300, 5)
  attack  epsilon             model      RMSE  run
0  clean      NaN              base  0.115062    1
1  clean      NaN  nov_bim_ensemble  0.177296    1
2    bim      0.1              base  0.250270    1
3    bim      0.1      nov_bim_rand  0.201571    1
4    bim      0.2              base  0.337186    1


In [ ]:
df_orig_bim = pd.read_csv(BIM_RUNS_CSV_ORIG)
df_nov_bim  = pd.read_csv(BIM_NOV_RUNS_30_CSV)

print("Loaded Morphence BIM runs:", df_orig_bim.shape)
print(df_orig_bim.head())
print("\nLoaded NOV BIM 30-run runs:", df_nov_bim.shape)
print(df_nov_bim.head())

COMBINED_BIM_DIR = BIM_NOV_RESULTS_DIR
os.makedirs(COMBINED_BIM_DIR, exist_ok=True)

Loaded Morphence BIM runs: (300, 5)
  attack  epsilon           model      RMSE  run
0  clean      NaN            base  0.115062    1
1  clean      NaN  morph_ensemble  0.158252    1
2    bim      0.1            base  0.249593    1
3    bim      0.1      morph_rand  0.192970    1
4    bim      0.2            base  0.338069    1

Loaded NOV BIM 30-run runs: (300, 5)
  attack  epsilon             model      RMSE  run
0  clean      NaN              base  0.115062    1
1  clean      NaN  nov_bim_ensemble  0.177296    1
2    bim      0.1              base  0.250270    1
3    bim      0.1      nov_bim_rand  0.201571    1
4    bim      0.2              base  0.337186    1


In [ ]:
df_clean_nov_bim = df_nov_bim[df_nov_bim["attack"] == "clean"].copy()

df_clean_nov_bim["model"] = pd.Categorical(
    df_clean_nov_bim["model"],
    categories=["base", "nov_bim_ensemble"],
    ordered=True
)

clean_nov_bim_stats_30 = (
    df_clean_nov_bim
    .groupby("model", observed=False)["RMSE"]
    .agg(
        mean   ="mean",
        std    ="std",
        min    ="min",
        q1     =lambda s: s.quantile(0.25),
        median ="median",
        q3     =lambda s: s.quantile(0.75),
        max    ="max",
        n      ="count",
    )
    .reset_index()
    .sort_values("model")
)

print("\n[NOV BIM | 30 runs | Clean] base vs nov_bim_ensemble:")
print(clean_nov_bim_stats_30.to_string(index=False))

clean_nov_bim_csv_30 = os.path.join(BIM_NOV_FIG_DIR, "bim_nov_clean_stats_30.csv")
clean_nov_bim_stats_30.to_csv(clean_nov_bim_csv_30, index=False)
print("Saved NOV BIM 30-run clean stats to:", clean_nov_bim_csv_30)

# BIM attack
# base vs nov_bim_rand per epsilon
df_bim_nov_attack = df_nov_bim[df_nov_bim["attack"] == "bim"].copy()

def nov_bim_model_stats(df, model_name, prefix="bim_nov_30"):
    st = (
        df[df["model"] == model_name]
        .groupby("epsilon", observed=False)["RMSE"]
        .agg(
            mean   ="mean",
            std    ="std",
            min    ="min",
            q1     =lambda s: s.quantile(0.25),
            median ="median",
            q3     =lambda s: s.quantile(0.75),
            max    ="max",
            n      ="count",
        )
        .reset_index()
        .sort_values("epsilon")
    )

    print(f"\n[NOV BIM | 30 runs | BIM] {model_name} stats:")
    print(st.to_string(index=False))

    out_csv = os.path.join(BIM_NOV_FIG_DIR, f"{prefix}_bim_{model_name}_stats.csv")
    st.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    return st

nov_bim_base_stats_30 = nov_bim_model_stats(df_bim_nov_attack, "base")
nov_bim_rand_stats_30 = nov_bim_model_stats(df_bim_nov_attack, "nov_bim_rand")


[NOV BIM | 30 runs | Clean] base vs nov_bim_ensemble:
           model     mean  std      min       q1   median       q3      max  n
            base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
nov_bim_ensemble 0.177296  0.0 0.177296 0.177296 0.177296 0.177296 0.177296 30
Saved NOV BIM 30-run clean stats to: /content/drive/MyDrive/298/Elec/NovTest/BIM/results/bim_nov_clean_stats_30.csv

[NOV BIM | 30 runs | BIM] base stats:
 epsilon     mean      std      min       q1   median       q3      max  n
     0.1 0.249272 0.000538 0.248380 0.248798 0.249220 0.249666 0.250270 30
     0.2 0.338730 0.000580 0.337186 0.338491 0.338754 0.339045 0.339870 30
     0.3 0.368153 0.000426 0.366911 0.367990 0.368186 0.368467 0.368885 30
     0.5 0.396055 0.000335 0.395389 0.395830 0.396004 0.396239 0.396747 30
Saved: /content/drive/MyDrive/298/Elec/NovTest/BIM/results/bim_nov_30_bim_base_stats.csv

[NOV BIM | 30 runs | BIM] nov_bim_rand stats:
 epsilon     mean      std      min       

In [ ]:
df_clean_orig_bim = df_orig_bim[df_orig_bim["attack"] == "clean"].copy()
df_clean_nov_bim  = df_nov_bim[df_nov_bim["attack"] == "clean"].copy()

# from Morphence BIM runs
# base + morph_ensemble
df_clean_orig_bim = df_clean_orig_bim[
    df_clean_orig_bim["model"].isin(["base", "morph_ensemble"])
]

# from NOV BIM runs
# nov_bim_ensemble
df_clean_nov_bim = df_clean_nov_bim[
    df_clean_nov_bim["model"].isin(["nov_bim_ensemble"])
]

df_clean_comb_bim = pd.concat([df_clean_orig_bim, df_clean_nov_bim], ignore_index=True)

df_clean_comb_bim["model"] = pd.Categorical(
    df_clean_comb_bim["model"],
    categories=["base", "morph_ensemble", "nov_bim_ensemble"],
    ordered=True
)

clean_comp_stats_bim = (
    df_clean_comb_bim
    .groupby("model", observed=False)["RMSE"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        q1=lambda s: s.quantile(0.25),
        median="median",
        q3=lambda s: s.quantile(0.75),
        max="max",
        n="count",
    )
    .reset_index()
    .sort_values("model")
)

print("\n[COMPARATIVE CLEAN | BIM] base vs morph_ensemble vs nov_bim_ensemble:")
print(clean_comp_stats_bim.to_string(index=False))

clean_comp_csv_bim = os.path.join(
    COMBINED_BIM_DIR, "bim_clean_stats_morphence_vs_nov_30runs.csv"
)
clean_comp_stats_bim.to_csv(clean_comp_csv_bim, index=False)
print("Saved comparative BIM clean stats to:", clean_comp_csv_bim)

df_bim_orig = df_orig_bim[df_orig_bim["attack"] == "bim"].copy()
df_bim_nov  = df_nov_bim[df_nov_bim["attack"] == "bim"].copy()

# from Morphence BIM runs
# base + morph_rand
df_bim_orig = df_bim_orig[
    df_bim_orig["model"].isin(["base", "morph_rand"])
]

# from NOV BIM runs
# nov_bim_rand
df_bim_nov = df_bim_nov[
    df_bim_nov["model"].isin(["nov_bim_rand"])
]

df_bim_comb = pd.concat([df_bim_orig, df_bim_nov], ignore_index=True)

df_bim_comb["model"] = pd.Categorical(
    df_bim_comb["model"],
    categories=["base", "morph_rand", "nov_bim_rand"],
    ordered=True
)

bim_comp_stats_30 = (
    df_bim_comb
    .groupby(["epsilon", "model"], observed=False)["RMSE"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        q1=lambda s: s.quantile(0.25),
        median="median",
        q3=lambda s: s.quantile(0.75),
        max="max",
        n="count",
    )
    .reset_index()
    .sort_values(["epsilon", "model"])
)

print("\n[COMPARATIVE BIM | 30 runs] base vs morph_rand vs nov_bim_rand:")
print(bim_comp_stats_30.to_string(index=False))

bim_comp_csv_30 = os.path.join(
    COMBINED_BIM_DIR, "bim_bim_stats_morphence_vs_nov_30runs.csv"
)
bim_comp_stats_30.to_csv(bim_comp_csv_30, index=False)
print("Saved comparative 30-run BIM stats to:", bim_comp_csv_30)


[COMPARATIVE CLEAN | BIM] base vs morph_ensemble vs nov_bim_ensemble:
           model     mean  std      min       q1   median       q3      max  n
            base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
  morph_ensemble 0.158252  0.0 0.158252 0.158252 0.158252 0.158252 0.158252 30
nov_bim_ensemble 0.177296  0.0 0.177296 0.177296 0.177296 0.177296 0.177296 30
Saved comparative BIM clean stats to: /content/drive/MyDrive/298/Elec/NovTest/BIM/results/bim_clean_stats_morphence_vs_nov_30runs.csv

[COMPARATIVE BIM | 30 runs] base vs morph_rand vs nov_bim_rand:
 epsilon        model     mean      std      min       q1   median       q3      max  n
     0.1         base 0.249131 0.000550 0.247888 0.248913 0.249107 0.249488 0.250070 30
     0.1   morph_rand 0.193049 0.000719 0.190970 0.192535 0.192961 0.193554 0.194659 30
     0.1 nov_bim_rand 0.200673 0.000445 0.199851 0.200347 0.200638 0.200972 0.201571 30
     0.2         base 0.338751 0.000457 0.337682 0.338551 0.33

In [ ]:
#pgd

In [ ]:
import os, traceback
import pandas as pd
import numpy as np
import torch
import torch.optim as optim

In [ ]:
NOV_PGD_PROJECT_DIR   = "/content/drive/MyDrive/298/Elec/NovTest/PGD"

PGD_STUDENTS_ADV_DIR  = os.path.join(NOV_PGD_PROJECT_DIR, "students_adv_nov_pgd")
PGD_RESULTS_DIR       = os.path.join(NOV_PGD_PROJECT_DIR, "results")
PGD_PLOTS_DIR         = os.path.join(PGD_RESULTS_DIR, "plots")
PGD_ARTIFACTS_DIR     = os.path.join(PGD_RESULTS_DIR, "artifacts")
PGD_CRASH_DIR         = os.path.join(NOV_PGD_PROJECT_DIR, "crash_dump")

PGD_MODEL_SIZES_CSV   = os.path.join(PGD_RESULTS_DIR, "pgd_nov_model_sizes.csv")
PGD_ERROR_LOG         = os.path.join(PGD_CRASH_DIR, "error_log.txt")

for d in [
    NOV_PGD_PROJECT_DIR,
    PGD_STUDENTS_ADV_DIR,
    PGD_RESULTS_DIR,
    PGD_PLOTS_DIR,
    PGD_ARTIFACTS_DIR,
    PGD_CRASH_DIR,
]:
    os.makedirs(d, exist_ok=True)

print("\n[PGD-Nov] NOV_PGD_PROJECT_DIR:", NOV_PGD_PROJECT_DIR)
print("[PGD-Nov] PGD_STUDENTS_ADV_DIR:", PGD_STUDENTS_ADV_DIR)
print("[PGD-Nov] PGD_RESULTS_DIR:", PGD_RESULTS_DIR)


[PGD-Nov] NOV_PGD_PROJECT_DIR: /content/drive/MyDrive/298/Elec/NovTest/PGD
[PGD-Nov] PGD_STUDENTS_ADV_DIR: /content/drive/MyDrive/298/Elec/NovTest/PGD/students_adv_nov_pgd
[PGD-Nov] PGD_RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/PGD/results


In [ ]:
#pgd hyperparams

PGD_TRAIN_EPS_LIST = [0.10, 0.20, 0.30, 0.50]
PGD_ITERS          = 10

PGD_RANDOM_START_TRAIN = True

def log_error_pgd(msg: str):
    with open(PGD_ERROR_LOG, "a") as f:
        f.write(msg + "\n")

In [ ]:
#pgd attack

def pgd_attack(attacker_model,
               X,
               Y,
               eps,
               alpha,
               iters,
               random_start=True,
               return_grad_norm=False):

    attacker_model.eval()

    if random_start:
        noise = torch.empty_like(X).uniform_(-eps, eps)
        X_adv = (X + noise).clamp(0.0, 1.0)
    else:
        X_adv = X.clone()

    X_adv = X_adv.detach().requires_grad_(True)
    last_grad = None

    for _ in range(iters):
        preds = attacker_model(X_adv)
        loss  = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad = X_adv.grad
        last_grad = grad

        X_adv = (X_adv + alpha * grad.sign()).detach()
        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach().requires_grad_(True)

    X_adv = X_adv.detach()
    if return_grad_norm and last_grad is not None:
        gn = last_grad.view(last_grad.size(0), -1).norm(p=2, dim=1)
        return X_adv, gn.detach()
    return X_adv


In [ ]:
#reconstruct novel students

NOV_STUDENTS_DIR = "/content/drive/MyDrive/298/Elec/NovTest/students_nov"

print("\n[PGD-Nov] Scanning for novel seed students in:", NOV_STUDENTS_DIR)

novel_student_paths = []

expected_map = {
    "student_05_enc_attn.pth": 5,
    "student_06_dec_attn.pth": 6,
    "student_07_ffn.pth": 7,
    "student_08_norms.pth": 8,
}

for fname in os.listdir(NOV_STUDENTS_DIR):
    if fname in expected_map:
        idx = expected_map[fname]
        name = fname.replace(".pth", "")
        full_path = os.path.join(NOV_STUDENTS_DIR, fname)
        novel_student_paths.append((idx, name, full_path))
        print(f"  Found NOV seed student {idx}: {name} -> {full_path}")

# sort by idx (5,6,7,8)
novel_student_paths = sorted(novel_student_paths, key=lambda x: x[0])

print("\n[PGD-Nov] Loaded novel seed students:")
for t in novel_student_paths:
    print(" ", t)

assert len(novel_student_paths) == 4, \
    f"[PGD-Nov] Expected 4 novel seed students (5–8), found {len(novel_student_paths)}"



[PGD-Nov] Scanning for novel seed students in: /content/drive/MyDrive/298/Elec/NovTest/students_nov
  Found NOV seed student 5: student_05_enc_attn -> /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_05_enc_attn.pth
  Found NOV seed student 6: student_06_dec_attn -> /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_06_dec_attn.pth
  Found NOV seed student 7: student_07_ffn -> /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_07_ffn.pth
  Found NOV seed student 8: student_08_norms -> /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_08_norms.pth

[PGD-Nov] Loaded novel seed students:
  (5, 'student_05_enc_attn', '/content/drive/MyDrive/298/Elec/NovTest/students_nov/student_05_enc_attn.pth')
  (6, 'student_06_dec_attn', '/content/drive/MyDrive/298/Elec/NovTest/students_nov/student_06_dec_attn.pth')
  (7, 'student_07_ffn', '/content/drive/MyDrive/298/Elec/NovTest/students_nov/student_07_ffn.pth')
  (8, 'student_08_norms', '/content/drive/MyDri

In [ ]:
#adv training of novel students

novel_pgd_adv_paths = []

print("\n[PGD-Nov] PGD adversarial training of NOV students")

for idx, name, seed_path in novel_student_paths:
    print(f"\n[PGD-Nov] [{name} | idx={idx}] Starting PGD adversarial training...")
    print(f"  Loading NOV seed from: {seed_path}")

    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(seed_path, map_location=device))

    opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)
    stu_history = []

    hb_flag, hb_thread = start_heartbeat(f"{name}_pgd_train")
    start_time = time.time()

    try:
        for ep in range(1, EPOCHS_STU + 1):
            stu.train()
            tot = cnt = 0

            for xb, yb in train_dl:
                xb = xb.to(device)
                yb = yb.to(device)

                # clean prediction loss
                pred_c = stu(xb)
                Lc = rmse_only(pred_c, yb)

                # multi-epsilon PGD adversarial loss
                adv_losses = []
                for eps in PGD_TRAIN_EPS_LIST:
                    alpha = eps / float(PGD_ITERS)
                    xa = pgd_attack(
                        attacker_model=stu,
                        X=xb,
                        Y=yb,
                        eps=eps,
                        alpha=alpha,
                        iters=PGD_ITERS,
                        random_start=PGD_RANDOM_START_TRAIN,
                        return_grad_norm=False
                    )
                    pred_a = stu(xa)
                    La = rmse_only(pred_a, yb)
                    adv_losses.append(La)

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                # same robust mix as FGSM/BIM
                # 0.25 clean + 0.75 adv
                loss = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot += loss.item() * bs
                cnt += bs

            avg_loss = tot / max(1, cnt)
            elapsed  = (time.time() - start_time) / 60.0
            print(
                f"  [PGD-Nov {name}] Epoch {ep:02d} | "
                f"avg_loss={avg_loss:.4f} | elapsed={elapsed:.2f} min",
                flush=True
            )

            stu_history.append({
                "student_idx": int(idx),
                "student_name": name,
                "epoch": ep,
                "avg_loss": avg_loss,
                "elapsed_min": elapsed,
            })

    except Exception as e:
        # on crash, stop heartbeat, log error, and save partial checkpoint
        stop_heartbeat(hb_flag, hb_thread)
        msg = (
            f"[PGD-Nov {name}] PGD adv training crash: {repr(e)}\n"
            f"{traceback.format_exc()}"
        )
        print(msg)
        log_error_pgd(msg)
        ckpt_crash = os.path.join(PGD_STUDENTS_ADV_DIR, f"{name}_pgd_CRASHED.pth")
        torch.save(stu.to("cpu").state_dict(), ckpt_crash)
        raise

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    # save per-student PGD training history CSV
    stu_hist_path = os.path.join(PGD_RESULTS_DIR, f"{name}_pgd_train_history.csv")
    df_stu_hist = pd.DataFrame(stu_history)
    df_stu_hist.to_csv(stu_hist_path, index=False)
    print(f"  [PGD-Nov {name}] Saved PGD training history to:", stu_hist_path)

    # save PGD-adv student checkpoint
    outp_pgd = os.path.join(PGD_STUDENTS_ADV_DIR, f"{name}_pgd_adv.pth")
    torch.save(stu.to("cpu").state_dict(), outp_pgd)
    novel_pgd_adv_paths.append((idx, name, outp_pgd))
    print(f"  [PGD-Nov {name}] Saved PGD adv student:", outp_pgd)

    # log PGD-adv student model size
    size_mb = model_file_size_mb(outp_pgd)
    df_ms = pd.DataFrame([{
        "model_name": f"{name}_pgd_adv",
        "path": outp_pgd,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, PGD_MODEL_SIZES_CSV)

# sort final list and print summary
novel_pgd_adv_paths = sorted(novel_pgd_adv_paths, key=lambda x: x[0])

print("\n[PGD-Nov] PGD adv trained NOV students:")
for idx, name, path in novel_pgd_adv_paths:
    print(f"  idx={idx} | {name}_pgd_adv - {path}")


[PGD-Nov] PGD adversarial training of NOV students

[PGD-Nov] [student_05_enc_attn | idx=5] Starting PGD adversarial training...
  Loading NOV seed from: /content/drive/MyDrive/298/Elec/NovTest/students_nov/student_05_enc_attn.pth


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[heartbeat:student_05_enc_attn_pgd_train] running... elapsed=10.00 min
[heartbeat:student_05_enc_attn_pgd_train] running... elapsed=20.00 min
  [PGD-Nov student_05_enc_attn] Epoch 01 | avg_loss=0.1444 | elapsed=23.23 min
[heartbeat:student_05_enc_attn_pgd_train] running... elapsed=30.00 min
[heartbeat:student_05_enc_attn_pgd_train] running... elapsed=40.00 min
  [PGD-Nov student_05_enc_attn] Epoch 02 | avg_loss=0.1411 | elapsed=46.59 min
[heartbeat:student_05_enc_attn_pgd_train] running... elapsed=50.00 min
[heartbeat:student_05_enc_attn_pgd_train] running... elapsed=60.00 min
  [PGD-Nov student_05_enc_attn] Epoch 03 | avg_loss=0.1384 | elapsed=69.90 min
[heartbeat:student_05_enc_attn_pgd_train] running... elapsed=70.00 min
[heartbeat:student_05_enc_attn_pgd_train] running... elapsed=80.00 min
[heartbeat:student_05_enc_attn_pgd_train] running... elapsed=90.00 min
  [PGD-Nov student_05_enc_attn] Epoch 04 | avg_loss=0.1380 | elapsed=93.23 min
[heartbeat:student_05_enc_attn_pgd_train] run

In [ ]:
import os
import numpy as np
import pandas as pd
import torch

In [ ]:
PGD_NOV_WEIGHT_DIVERSITY_CSV = os.path.join(PGD_RESULTS_DIR, "pgd_nov_weight_diversity.csv")
PGD_NOV_PRED_DIVERSITY_CSV   = os.path.join(PGD_RESULTS_DIR, "pgd_nov_prediction_diversity.csv")
PGD_NOV_XFER_MATRIX_CSV      = os.path.join(PGD_RESULTS_DIR, "pgd_nov_transfer_matrix.csv")

In [ ]:
PGD_EVAL_EPS_LIST     = [0.10, 0.20, 0.30, 0.50]
PGD_RANDOM_START_EVAL = True

In [ ]:
print("\n[PGD-Nov] Loading PGD adversarially-trained NOV students...")

nov_pgd_paths = sorted(
    [
        os.path.join(PGD_STUDENTS_ADV_DIR, f)
        for f in os.listdir(PGD_STUDENTS_ADV_DIR)
        if f.endswith("_pgd_adv.pth")
    ]
)

assert len(nov_pgd_paths) == 4, \
    f"[PGD-Nov] Expected 4 PGD adv students (5–8). Found {len(nov_pgd_paths)}."

nov_pgd_students = []
nov_pgd_names    = []

for p in nov_pgd_paths:
    name = os.path.basename(p).replace(".pth", "")
    print("  Loading:", name)

    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)

    stu.load_state_dict(torch.load(p, map_location=device))
    stu.eval()

    nov_pgd_students.append(stu)
    nov_pgd_names.append(name)

print("\n[PGD-Nov] Loaded PGD NOV students:")
for n in nov_pgd_names:
    print(" ", n)


[PGD-Nov] Loading PGD adversarially-trained NOV students...
  Loading: student_05_enc_attn_pgd_adv


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  Loading: student_06_dec_attn_pgd_adv
  Loading: student_07_ffn_pgd_adv
  Loading: student_08_norms_pgd_adv

[PGD-Nov] Loaded PGD NOV students:
  student_05_enc_attn_pgd_adv
  student_06_dec_attn_pgd_adv
  student_07_ffn_pgd_adv
  student_08_norms_pgd_adv


In [ ]:
def flatten_params(model):
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])

def compute_weight_diversity_pgd(students, names, out_csv=PGD_NOV_WEIGHT_DIVERSITY_CSV):
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)
    rows = []
    for i in range(k):
        for j in range(i + 1, k):
            d = torch.norm(vecs[i] - vecs[j], p=2).item()
            rows.append({
                "student_i": names[i],
                "student_j": names[j],
                "l2_distance": d,
            })
    df_w = pd.DataFrame(rows)
    df_w.to_csv(out_csv, index=False)
    print("[PGD-Nov] Saved weight diversity to:", out_csv)
    return df_w

@torch.no_grad()
def compute_prediction_diversity_pgd(students, loader, out_csv=PGD_NOV_PRED_DIVERSITY_CSV):
    for m in students:
        m.eval()

    k = len(students)
    H = HZN_STEPS

    sum_pred  = np.zeros(H, dtype=np.float64)
    sum_pred2 = np.zeros(H, dtype=np.float64)
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device)
        preds_stu = []
        for m in students:
            preds_stu.append(m(xb).cpu().numpy())  # (B,H,1)
        preds_stu = np.stack(preds_stu, axis=0)    # (k,B,H,1)

        # mean prediction per student over batch, then average over students
        preds_mean_batch = preds_stu.mean(axis=1).squeeze(-1)  # (k,H)

        sum_pred  += preds_mean_batch.mean(axis=0)
        sum_pred2 += (preds_mean_batch ** 2).mean(axis=0)
        count     += 1

    mean_pred  = sum_pred / max(1, count)
    mean_pred2 = sum_pred2 / max(1, count)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "horizon_idx": np.arange(H),
        "mean_pred": mean_pred,
        "var_pred": var_pred,
    })
    df_p.to_csv(out_csv, index=False)
    print("[PGD-Nov] Saved prediction diversity to:", out_csv)
    return df_p

print("\n[PGD-Nov] Computing diversity metrics for PGD trained novel students...")
_ = compute_weight_diversity_pgd(nov_pgd_students, nov_pgd_names)
_ = compute_prediction_diversity_pgd(nov_pgd_students, test_dl)



[PGD-Nov] Computing diversity metrics for PGD-trained novel students...
[PGD-Nov] Saved weight diversity to: /content/drive/MyDrive/298/Elec/NovTest/PGD/results/pgd_nov_weight_diversity.csv
[PGD-Nov] Saved prediction diversity to: /content/drive/MyDrive/298/Elec/NovTest/PGD/results/pgd_nov_prediction_diversity.csv


In [ ]:
def eval_pair_pgd(defender,
                  attacker,
                  loader,
                  eps,
                  iters=PGD_ITERS,
                  random_start=PGD_RANDOM_START_EVAL):
    """
    Evaluate defender under PGD attack crafted by attacker.
    Uses PGD with L_inf budget eps and 'iters' steps.
    """
    defender.eval()
    attacker.eval()

    ys, ps = [], []
    alpha = eps / float(iters)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        # Craft PGD adversarial inputs with the attacker (needs grads)
        xa = pgd_attack(
            attacker_model=attacker,
            X=xb,
            Y=yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
            return_grad_norm=False,
        )

        # Evaluate defender on the crafted adversarial examples (no grads needed)
        with torch.no_grad():
            p = defender(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
xfer_rows_pgd_nov = []

print("\n[PGD-Nov] Computing NOV PGD transfer matrix (PGD attack)...")
for eps in PGD_EVAL_EPS_LIST:
    print(f"  eps={eps:.2f} ...")
    for i, atk_name in enumerate(nov_pgd_names):
        for j, def_name in enumerate(nov_pgd_names):
            atk_model = nov_pgd_students[i]
            def_model = nov_pgd_students[j]

            rmse_ij = eval_pair_pgd(
                defender=def_model,
                attacker=atk_model,
                loader=test_dl,
                eps=eps,
                iters=PGD_ITERS,
                random_start=PGD_RANDOM_START_EVAL,
            )

            xfer_rows_pgd_nov.append({
                "epsilon": eps,
                "attacker": atk_name,
                "defender": def_name,
                "rmse_adv": rmse_ij,
            })
            print(f"    atk={atk_name} -> def={def_name} | eps={eps:.2f} | RMSE={rmse_ij:.5f}")

df_xfer_pgd_nov = pd.DataFrame(xfer_rows_pgd_nov)
df_xfer_pgd_nov.to_csv(PGD_NOV_XFER_MATRIX_CSV, index=False)
print("\n[PGD-Nov] Saved NOV-PGD transfer matrix to:", PGD_NOV_XFER_MATRIX_CSV)


[PGD-Nov] Computing NOV PGD transfer matrix (PGD attack)...
  eps=0.10 ...
    atk=student_05_enc_attn_pgd_adv -> def=student_05_enc_attn_pgd_adv | eps=0.10 | RMSE=0.19634
    atk=student_05_enc_attn_pgd_adv -> def=student_06_dec_attn_pgd_adv | eps=0.10 | RMSE=0.18377
    atk=student_05_enc_attn_pgd_adv -> def=student_07_ffn_pgd_adv | eps=0.10 | RMSE=0.19759
    atk=student_05_enc_attn_pgd_adv -> def=student_08_norms_pgd_adv | eps=0.10 | RMSE=0.18021
    atk=student_06_dec_attn_pgd_adv -> def=student_05_enc_attn_pgd_adv | eps=0.10 | RMSE=0.19628
    atk=student_06_dec_attn_pgd_adv -> def=student_06_dec_attn_pgd_adv | eps=0.10 | RMSE=0.18435
    atk=student_06_dec_attn_pgd_adv -> def=student_07_ffn_pgd_adv | eps=0.10 | RMSE=0.19771
    atk=student_06_dec_attn_pgd_adv -> def=student_08_norms_pgd_adv | eps=0.10 | RMSE=0.18098
    atk=student_07_ffn_pgd_adv -> def=student_05_enc_attn_pgd_adv | eps=0.10 | RMSE=0.19630
    atk=student_07_ffn_pgd_adv -> def=student_06_dec_attn_pgd_adv | eps=

In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import torch

In [ ]:
PGD_NOV_RUNS_30_CSV  = os.path.join(PGD_RESULTS_DIR, "pgd_nov_runs_30.csv")
PGD_NOV_STATS_30_CSV = os.path.join(PGD_RESULTS_DIR, "pgd_nov_stats_30.csv")

print("\n[PGD-Nov Eval] PGD_RESULTS_DIR:", PGD_RESULTS_DIR)
print("[PGD-Nov Eval] PGD_NOV_RUNS_30_CSV:", PGD_NOV_RUNS_30_CSV)


[PGD-Nov Eval] PGD_RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/PGD/results
[PGD-Nov Eval] PGD_NOV_RUNS_30_CSV: /content/drive/MyDrive/298/Elec/NovTest/PGD/results/pgd_nov_runs_30.csv


In [ ]:
best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(BASE_CKPT, map_location=device))
best_base.eval()
print("[PGD-Nov Eval] Loaded base model from:", BASE_CKPT)


[PGD-Nov Eval] Loaded base model from: /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth


In [ ]:
nov_pgd_paths = sorted(
    [
        os.path.join(PGD_STUDENTS_ADV_DIR, f)
        for f in os.listdir(PGD_STUDENTS_ADV_DIR)
        if f.endswith("_pgd_adv.pth")
    ]
)

assert len(nov_pgd_paths) > 0, "[PGD-Nov Eval] No *_pgd_adv.pth found in PGD_STUDENTS_ADV_DIR!"
print("[PGD-Nov Eval] NOV PGD students:")
for p in nov_pgd_paths:
    print("  ", p)

nov_pgd_students = []
nov_pgd_names    = []
for sp in nov_pgd_paths:
    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))
    stu.eval()
    nov_pgd_students.append(stu)
    nov_pgd_names.append(os.path.basename(sp).replace(".pth", ""))

print(f"[PGD-Nov Eval] Loaded {len(nov_pgd_students)} NOV PGD students.")


[PGD-Nov Eval] NOV PGD students:
   /content/drive/MyDrive/298/Elec/NovTest/PGD/students_adv_nov_pgd/student_05_enc_attn_pgd_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/PGD/students_adv_nov_pgd/student_06_dec_attn_pgd_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/PGD/students_adv_nov_pgd/student_07_ffn_pgd_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/PGD/students_adv_nov_pgd/student_08_norms_pgd_adv.pth
[PGD-Nov Eval] Loaded 4 NOV PGD students.


In [ ]:
@torch.no_grad()
def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        p  = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

@torch.no_grad()
def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        preds = []
        for m in models:
            preds.append(m(xb))
        preds = torch.stack(preds, dim=0).mean(dim=0)  # (B,H,D)
        ys.append(yb.cpu().numpy())
        ps.append(preds.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_pair_pgd(defender,
                  attacker,
                  loader,
                  eps,
                  iters=PGD_ITERS,
                  random_start=PGD_RANDOM_START_EVAL):
    """
    Evaluate defender under PGD attack crafted by attacker.
    """
    defender.eval()
    attacker.eval()

    ys, ps = [], []
    alpha = eps / float(iters)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        xa = pgd_attack(
            attacker_model=attacker,
            X=xb,
            Y=yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
            return_grad_norm=False,
        )

        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
def eval_random_switch_pgd(models,
                           loader,
                           eps,
                           iters=PGD_ITERS,
                           random_start=PGD_RANDOM_START_EVAL):
    """
    Randomly pick attacker+defender student per batch (Morphence-style).
    """
    if len(models) == 0:
        raise ValueError("No models provided to eval_random_switch_pgd")

    for m in models:
        m.eval()

    ys, ps = [], []
    n_stu = len(models)
    alpha = eps / float(iters)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        atk_idx  = random.randrange(n_stu)
        def_idx  = random.randrange(n_stu)
        attacker = models[atk_idx]
        defender = models[def_idx]

        xa = pgd_attack(
            attacker_model=attacker,
            X=xb,
            Y=yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
            return_grad_norm=False,
        )

        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
N_RUNS_LARGE = 30

print("\n[PGD-Nov Eval] 30-run PGD eval: base vs NOV PGD ensemble")
print("  Epsilons:", PGD_EVAL_EPS_LIST)
print("  Runs:", N_RUNS_LARGE)

all_rows_pgd_nov = []   # ["attack", "epsilon", "model", "RMSE", "run"]

start_all_30 = time.time()
hb_flag_30, hb_thread_30 = start_heartbeat("eval_nov_pgd_30runs")

try:
    for run_i in range(1, N_RUNS_LARGE + 1):
        run_start = time.time()
        seed_i = 12000 + run_i
        random.seed(seed_i)
        np.random.seed(seed_i)
        torch.manual_seed(seed_i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed_i)

        # Clean RMSE
        base_clean = eval_clean_rmse(best_base,        test_dl)
        nov_clean  = eval_ensemble_rmse(nov_pgd_students, test_dl)

        all_rows_pgd_nov.append(["clean", None, "base",             base_clean, run_i])
        all_rows_pgd_nov.append(["clean", None, "nov_pgd_ensemble", nov_clean,  run_i])

        print(
            f"[PGD-Nov Run {run_i:02d}] clean | "
            f"base={base_clean:.5f} | nov_pgd_ens={nov_clean:.5f}",
            flush=True
        )

        # PGD attack per epsilon
        for eps in PGD_EVAL_EPS_LIST:
            base_adv = eval_pair_pgd(
                defender=best_base,
                attacker=best_base,
                loader=test_dl,
                eps=eps,
            )

            nov_rand = eval_random_switch_pgd(
                models=nov_pgd_students,
                loader=test_dl,
                eps=eps,
            )

            all_rows_pgd_nov.append(["pgd", eps, "base",          base_adv, run_i])
            all_rows_pgd_nov.append(["pgd", eps, "nov_pgd_rand",  nov_rand, run_i])

            print(
                f"[PGD-Nov Run {run_i:02d}] eps={eps:.2f} | "
                f"base_adv={base_adv:.5f} | nov_pgd_rand={nov_rand:.5f}",
                flush=True
            )

        # Autosave after each run
        df_runs_pgd_nov = pd.DataFrame(
            all_rows_pgd_nov,
            columns=["attack", "epsilon", "model", "RMSE", "run"]
        )
        df_runs_pgd_nov.to_csv(PGD_NOV_RUNS_30_CSV, index=False)

        stats_pgd_nov = (
            df_runs_pgd_nov
            .groupby(["attack", "epsilon", "model"])["RMSE"]
            .agg(mean="mean", std="std", n="count")
            .reset_index()
            .sort_values(["attack", "epsilon", "model"])
        )
        stats_pgd_nov.to_csv(PGD_NOV_STATS_30_CSV, index=False)

        run_mins   = (time.time() - run_start) / 60.0
        total_mins = (time.time() - start_all_30) / 60.0
        print(
            f"[PGD-Nov Eval] Completed run {run_i}/{N_RUNS_LARGE} | "
            f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
            flush=True
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()

finally:
    stop_heartbeat(hb_flag_30, hb_thread_30)

print("\n[PGD-Nov Eval] Saved 30-run NOV PGD runs CSV to:", PGD_NOV_RUNS_30_CSV)
print("[PGD-Nov Eval] Saved 30-run NOV PGD stats CSV to:", PGD_NOV_STATS_30_CSV)


[PGD-Nov Eval] 30-run PGD eval: base vs NOV PGD ensemble
  Epsilons: [0.1, 0.2, 0.3, 0.5]
  Runs: 30
[PGD-Nov Run 01] clean | base=0.11506 | nov_pgd_ens=0.18829
[PGD-Nov Run 01] eps=0.10 | base_adv=0.24899 | nov_pgd_rand=0.18995
[PGD-Nov Run 01] eps=0.20 | base_adv=0.33900 | nov_pgd_rand=0.18916
[PGD-Nov Run 01] eps=0.30 | base_adv=0.36804 | nov_pgd_rand=0.19587
[heartbeat:eval_nov_pgd_30runs] running... elapsed=10.00 min
[PGD-Nov Run 01] eps=0.50 | base_adv=0.39673 | nov_pgd_rand=0.22061
[PGD-Nov Eval] Completed run 1/30 | run_time=12.46 min | total_elapsed=12.46 min
[PGD-Nov Run 02] clean | base=0.11506 | nov_pgd_ens=0.18829
[PGD-Nov Run 02] eps=0.10 | base_adv=0.24955 | nov_pgd_rand=0.18911
[PGD-Nov Run 02] eps=0.20 | base_adv=0.33873 | nov_pgd_rand=0.19133
[heartbeat:eval_nov_pgd_30runs] running... elapsed=20.00 min
[PGD-Nov Run 02] eps=0.30 | base_adv=0.36882 | nov_pgd_rand=0.19514
[PGD-Nov Run 02] eps=0.50 | base_adv=0.39569 | nov_pgd_rand=0.21889
[PGD-Nov Eval] Completed run 2/

In [ ]:
#runtime crashed
#need a resume logic

In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import torch

In [ ]:
PGD_NOV_RUNS_30_CSV  = os.path.join(PGD_RESULTS_DIR, "pgd_nov_runs_30.csv")
PGD_NOV_STATS_30_CSV = os.path.join(PGD_RESULTS_DIR, "pgd_nov_stats_30.csv")

print("\n[PGD-Nov Eval] PGD_RESULTS_DIR:", PGD_RESULTS_DIR)
print("[PGD-Nov Eval] PGD_NOV_RUNS_30_CSV:", PGD_NOV_RUNS_30_CSV)


[PGD-Nov Eval] PGD_RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/PGD/results
[PGD-Nov Eval] PGD_NOV_RUNS_30_CSV: /content/drive/MyDrive/298/Elec/NovTest/PGD/results/pgd_nov_runs_30.csv


In [ ]:
best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(BASE_CKPT, map_location=device))
best_base.eval()
print("[PGD-Nov Eval] Loaded base model from:", BASE_CKPT)


[PGD-Nov Eval] Loaded base model from: /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth


In [ ]:
# reload NOV-PGD students
nov_pgd_paths = sorted(
    [
        os.path.join(PGD_STUDENTS_ADV_DIR, f)
        for f in os.listdir(PGD_STUDENTS_ADV_DIR)
        if f.endswith("_pgd_adv.pth")
    ]
)

assert len(nov_pgd_paths) > 0, "[PGD-Nov Eval] No *_pgd_adv.pth found in PGD_STUDENTS_ADV_DIR!"
print("[PGD-Nov Eval] NOV PGD students:")
for p in nov_pgd_paths:
    print("  ", p)

nov_pgd_students = []
nov_pgd_names    = []
for sp in nov_pgd_paths:
    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))
    stu.eval()
    nov_pgd_students.append(stu)
    nov_pgd_names.append(os.path.basename(sp).replace(".pth", ""))

print(f"[PGD-Nov Eval] Loaded {len(nov_pgd_students)} NOV PGD students.")

[PGD-Nov Eval] NOV PGD students:
   /content/drive/MyDrive/298/Elec/NovTest/PGD/students_adv_nov_pgd/student_05_enc_attn_pgd_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/PGD/students_adv_nov_pgd/student_06_dec_attn_pgd_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/PGD/students_adv_nov_pgd/student_07_ffn_pgd_adv.pth
   /content/drive/MyDrive/298/Elec/NovTest/PGD/students_adv_nov_pgd/student_08_norms_pgd_adv.pth
[PGD-Nov Eval] Loaded 4 NOV PGD students.


In [ ]:
# basic eval helpers

@torch.no_grad()
def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        p  = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

@torch.no_grad()
def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        preds = []
        for m in models:
            preds.append(m(xb))
        preds = torch.stack(preds, dim=0).mean(dim=0)  # (B,H,D)
        ys.append(yb.cpu().numpy())
        ps.append(preds.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_pair_pgd(defender,
                  attacker,
                  loader,
                  eps,
                  iters=PGD_ITERS,
                  random_start=PGD_RANDOM_START_EVAL):
    """
    Evaluate defender under PGD attack crafted by attacker.
    """
    defender.eval()
    attacker.eval()

    ys, ps = [], []
    alpha = eps / float(iters)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        xa = pgd_attack(
            attacker_model=attacker,
            X=xb,
            Y=yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
            return_grad_norm=False,
        )

        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_random_switch_pgd(models,
                           loader,
                           eps,
                           iters=PGD_ITERS,
                           random_start=PGD_RANDOM_START_EVAL):
    """
    Randomly pick attacker+defender student per batch (Morphence-style).
    """
    if len(models) == 0:
        raise ValueError("No models provided to eval_random_switch_pgd")

    for m in models:
        m.eval()

    ys, ps = [], []
    n_stu = len(models)
    alpha = eps / float(iters)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        atk_idx  = random.randrange(n_stu)
        def_idx  = random.randrange(n_stu)
        attacker = models[atk_idx]
        defender = models[def_idx]

        xa = pgd_attack(
            attacker_model=attacker,
            X=xb,
            Y=yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
            return_grad_norm=False,
        )

        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
# RESUME LOGIC

N_RUNS_LARGE = 30
ROWS_PER_RUN = 10

all_rows_pgd_nov = []
start_run = 1

if os.path.exists(PGD_NOV_RUNS_30_CSV):
    print("\n[PGD-Nov Eval] Existing runs file found, loading to resume...")
    df_existing = pd.read_csv(PGD_NOV_RUNS_30_CSV)

    if "run" in df_existing.columns:
        rows_per_run = df_existing.groupby("run").size()
        print("[PGD-Nov Eval] Rows per run so far:")
        print(rows_per_run)

        full_runs = rows_per_run[rows_per_run == ROWS_PER_RUN].index.tolist()
        partial_runs = rows_per_run[rows_per_run != ROWS_PER_RUN].index.tolist()

        if partial_runs:
            print("[PGD-Nov Eval] Dropping partial runs:", partial_runs)
            df_existing = df_existing[~df_existing["run"].isin(partial_runs)]

        if len(full_runs) > 0:
            last_full_run = max(full_runs)
            start_run = last_full_run + 1
            print(f"[PGD-Nov Eval] Last full run: {last_full_run}. Resuming from run {start_run}.")
            all_rows_pgd_nov = df_existing[["attack", "epsilon", "model", "RMSE", "run"]].values.tolist()
        else:
            print("[PGD-Nov Eval] No full runs detected. Starting from scratch.")
            start_run = 1
            all_rows_pgd_nov = []
    else:
        print("[PGD-Nov Eval] 'run' column missing in existing CSV. Starting from scratch.")
        start_run = 1
        all_rows_pgd_nov = []
else:
    print("\n[PGD-Nov Eval] No existing runs CSV. Starting from run 1.")
    start_run = 1
    all_rows_pgd_nov = []

if start_run > N_RUNS_LARGE:
    print("\n[PGD-Nov Eval] All 30 runs already completed. Recomputing stats and exiting.")
    df_runs_pgd_nov = pd.DataFrame(
        all_rows_pgd_nov,
        columns=["attack", "epsilon", "model", "RMSE", "run"]
    )
    stats_pgd_nov = (
        df_runs_pgd_nov
        .groupby(["attack", "epsilon", "model"])["RMSE"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
        .sort_values(["attack", "epsilon", "model"])
    )
    stats_pgd_nov.to_csv(PGD_NOV_STATS_30_CSV, index=False)
    print("[PGD-Nov Eval] Stats recomputed and saved to:", PGD_NOV_STATS_30_CSV)
else:
    print("\n[PGD-Nov Eval] 30-run PGD eval: base vs NOV PGD ensemble")
    print("  Epsilons:", PGD_EVAL_EPS_LIST)
    print("  Runs:", N_RUNS_LARGE)
    print("  Starting from run:", start_run)

    start_all_30 = time.time()
    hb_flag_30, hb_thread_30 = start_heartbeat("eval_nov_pgd_30runs")

    try:
        for run_i in range(start_run, N_RUNS_LARGE + 1):
            run_start = time.time()
            seed_i = 12000 + run_i
            random.seed(seed_i)
            np.random.seed(seed_i)
            torch.manual_seed(seed_i)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed_i)

            # clean RMSE
            base_clean = eval_clean_rmse(best_base,          test_dl)
            nov_clean  = eval_ensemble_rmse(nov_pgd_students, test_dl)

            all_rows_pgd_nov.append(["clean", None, "base",             base_clean, run_i])
            all_rows_pgd_nov.append(["clean", None, "nov_pgd_ensemble", nov_clean,  run_i])

            print(
                f"[PGD-Nov Run {run_i:02d}] clean | "
                f"base={base_clean:.5f} | nov_pgd_ens={nov_clean:.5f}",
                flush=True
            )

            # pGD attack per epsilon
            for eps in PGD_EVAL_EPS_LIST:
                base_adv = eval_pair_pgd(
                    defender=best_base,
                    attacker=best_base,
                    loader=test_dl,
                    eps=eps,
                )

                nov_rand = eval_random_switch_pgd(
                    models=nov_pgd_students,
                    loader=test_dl,
                    eps=eps,
                )

                all_rows_pgd_nov.append(["pgd", eps, "base",         base_adv, run_i])
                all_rows_pgd_nov.append(["pgd", eps, "nov_pgd_rand", nov_rand, run_i])

                print(
                    f"[PGD-Nov Run {run_i:02d}] eps={eps:.2f} | "
                    f"base_adv={base_adv:.5f} | nov_pgd_rand={nov_rand:.5f}",
                    flush=True
                )

            # autosave after each run
            df_runs_pgd_nov = pd.DataFrame(
                all_rows_pgd_nov,
                columns=["attack", "epsilon", "model", "RMSE", "run"]
            )
            df_runs_pgd_nov.to_csv(PGD_NOV_RUNS_30_CSV, index=False)

            stats_pgd_nov = (
                df_runs_pgd_nov
                .groupby(["attack", "epsilon", "model"])["RMSE"]
                .agg(mean="mean", std="std", n="count")
                .reset_index()
                .sort_values(["attack", "epsilon", "model"])
            )
            stats_pgd_nov.to_csv(PGD_NOV_STATS_30_CSV, index=False)

            run_mins   = (time.time() - run_start) / 60.0
            total_mins = (time.time() - start_all_30) / 60.0
            print(
                f"[PGD-Nov Eval] Completed run {run_i}/{N_RUNS_LARGE} | "
                f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
                flush=True
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
            gc.collect()

    finally:
        stop_heartbeat(hb_flag_30, hb_thread_30)

    print("\n[PGD-Nov Eval] Saved 30-run NOV PGD runs CSV to:", PGD_NOV_RUNS_30_CSV)
    print("[PGD-Nov Eval] Saved 30-run NOV PGD stats CSV to:", PGD_NOV_STATS_30_CSV)


[PGD-Nov Eval] Existing runs file found, loading to resume...
[PGD-Nov Eval] Rows per run so far:
run
1     10
2     10
3     10
4     10
5     10
6     10
7     10
8     10
9     10
10    10
11    10
12    10
13    10
14    10
15    10
16    10
17    10
18    10
19    10
20    10
21    10
22    10
23    10
24    10
25    10
26    10
27    10
dtype: int64
[PGD-Nov Eval] Last full run: 27. Resuming from run 28.

[PGD-Nov Eval] 30-run PGD eval: base vs NOV PGD ensemble
  Epsilons: [0.1, 0.2, 0.3, 0.5]
  Runs: 30
  Starting from run: 28
[PGD-Nov Run 28] clean | base=0.11506 | nov_pgd_ens=0.18829
[PGD-Nov Run 28] eps=0.10 | base_adv=0.24953 | nov_pgd_rand=0.18964
[PGD-Nov Run 28] eps=0.20 | base_adv=0.33920 | nov_pgd_rand=0.19191
[PGD-Nov Run 28] eps=0.30 | base_adv=0.36903 | nov_pgd_rand=0.19544
[heartbeat:eval_nov_pgd_30runs] running... elapsed=10.00 min
[PGD-Nov Run 28] eps=0.50 | base_adv=0.39636 | nov_pgd_rand=0.21939
[PGD-Nov Eval] Completed run 28/30 | run_time=11.59 min | total_el

In [ ]:
#PGD stats

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
NOV_PGD_PROJECT_DIR   = "/content/drive/MyDrive/298/Elec/NovTest/PGD"
NOV_PGD_RESULTS_DIR   = os.path.join(NOV_PGD_PROJECT_DIR, "results")
NOV_PGD_FIG_DIR       = NOV_PGD_RESULTS_DIR

PGD_NOV_RUNS_30_CSV   = os.path.join(NOV_PGD_RESULTS_DIR, "pgd_nov_runs_30.csv")

print("NOV_PGD_RESULTS_DIR:", NOV_PGD_RESULTS_DIR)
print("PGD_NOV_RUNS_30_CSV:", PGD_NOV_RUNS_30_CSV)
os.makedirs(NOV_PGD_FIG_DIR, exist_ok=True)


NOV_PGD_RESULTS_DIR: /content/drive/MyDrive/298/Elec/NovTest/PGD/results
PGD_NOV_RUNS_30_CSV: /content/drive/MyDrive/298/Elec/NovTest/PGD/results/pgd_nov_runs_30.csv


In [ ]:
df_nov_pgd = pd.read_csv(PGD_NOV_RUNS_30_CSV)
print("Loaded NOV PGD 30-run runs:", df_nov_pgd.shape)
print(df_nov_pgd.head())

Loaded NOV PGD 30-run runs: (300, 5)
  attack  epsilon             model      RMSE  run
0  clean      NaN              base  0.115062    1
1  clean      NaN  nov_pgd_ensemble  0.188289    1
2    pgd      0.1              base  0.248992    1
3    pgd      0.1      nov_pgd_rand  0.189954    1
4    pgd      0.2              base  0.338999    1


In [ ]:
df_clean_nov = df_nov_pgd[df_nov_pgd["attack"] == "clean"].copy()

df_clean_nov["model"] = pd.Categorical(
    df_clean_nov["model"],
    categories=["base", "nov_pgd_ensemble"],
    ordered=True
)

clean_nov_stats_30 = (
    df_clean_nov
    .groupby("model", observed=False)["RMSE"]
    .agg(
        mean   ="mean",
        std    ="std",
        min    ="min",
        q1     =lambda s: s.quantile(0.25),
        median ="median",
        q3     =lambda s: s.quantile(0.75),
        max    ="max",
        n      ="count",
    )
    .reset_index()
    .sort_values("model")
)

print("\n[NOV PGD | 30 runs | Clean] base vs nov_pgd_ensemble:")
print(clean_nov_stats_30.to_string(index=False))

clean_nov_csv_30 = os.path.join(NOV_PGD_FIG_DIR, "pgd_nov_clean_stats_30.csv")
clean_nov_stats_30.to_csv(clean_nov_csv_30, index=False)
print("Saved NOV PGD 30-run clean stats to:", clean_nov_csv_30)


df_pgd_nov_attack = df_nov_pgd[df_nov_pgd["attack"] == "pgd"].copy()

def nov_pgd_model_stats(df, model_name, prefix="pgd_nov_30"):
    st = (
        df[df["model"] == model_name]
        .groupby("epsilon", observed=False)["RMSE"]
        .agg(
            mean   ="mean",
            std    ="std",
            min    ="min",
            q1     =lambda s: s.quantile(0.25),
            median ="median",
            q3     =lambda s: s.quantile(0.75),
            max    ="max",
            n      ="count",
        )
        .reset_index()
        .sort_values("epsilon")
    )

    print(f"\n[NOV PGD | 30 runs | PGD] {model_name} stats:")
    print(st.to_string(index=False))

    out_csv = os.path.join(NOV_PGD_FIG_DIR, f"{prefix}_pgd_{model_name}_stats.csv")
    st.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    return st

nov_base_stats_30     = nov_pgd_model_stats(df_pgd_nov_attack, "base")
nov_rand_stats_30     = nov_pgd_model_stats(df_pgd_nov_attack, "nov_pgd_rand")



[NOV PGD | 30 runs | Clean] base vs nov_pgd_ensemble:
           model     mean  std      min       q1   median       q3      max  n
            base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
nov_pgd_ensemble 0.188289  0.0 0.188289 0.188289 0.188289 0.188289 0.188289 30
Saved NOV PGD 30-run clean stats to: /content/drive/MyDrive/298/Elec/NovTest/PGD/results/pgd_nov_clean_stats_30.csv

[NOV PGD | 30 runs | PGD] base stats:
 epsilon     mean      std      min       q1   median       q3      max  n
     0.1 0.249299 0.000526 0.248310 0.249011 0.249246 0.249527 0.250548 30
     0.2 0.338715 0.000409 0.337686 0.338515 0.338711 0.339057 0.339404 30
     0.3 0.368249 0.000460 0.367396 0.367984 0.368152 0.368583 0.369079 30
     0.5 0.396025 0.000451 0.395253 0.395651 0.396070 0.396356 0.396726 30
Saved: /content/drive/MyDrive/298/Elec/NovTest/PGD/results/pgd_nov_30_pgd_base_stats.csv

[NOV PGD | 30 runs | PGD] nov_pgd_rand stats:
 epsilon     mean      std      min       

In [ ]:
ORIG_PGD_RESULTS_DIR  = "/content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results"
PGD_ORIG_RUNS_CSV     = os.path.join(ORIG_PGD_RESULTS_DIR, "pgd_morphence_runs.csv")

print("\nORIG_PGD_RESULTS_DIR:", ORIG_PGD_RESULTS_DIR)
print("PGD_ORIG_RUNS_CSV:", PGD_ORIG_RUNS_CSV)

df_orig_pgd = pd.read_csv(PGD_ORIG_RUNS_CSV)
print("Loaded original PGD runs:", df_orig_pgd.shape)
print(df_orig_pgd.head())


ORIG_PGD_RESULTS_DIR: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results
PGD_ORIG_RUNS_CSV: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results/pgd_morphence_runs.csv
Loaded original PGD runs: (300, 5)
  attack  epsilon           model      RMSE  run
0  clean      NaN            base  0.115062    1
1  clean      NaN  morph_ensemble  0.191335    1
2    pgd      0.1            base  0.249135    1
3    pgd      0.1      morph_rand  0.193250    1
4    pgd      0.2            base  0.338785    1


In [ ]:
df_clean_orig = df_orig_pgd[df_orig_pgd["attack"] == "clean"].copy()
df_clean_orig = df_clean_orig[df_clean_orig["model"].isin(["base", "morph_ensemble"])]

df_clean_orig["model"] = pd.Categorical(
    df_clean_orig["model"],
    categories=["base", "morph_ensemble"],
    ordered=True
)

clean_orig_stats_30 = (
    df_clean_orig
    .groupby("model", observed=False)["RMSE"]
    .agg(
        mean   ="mean",
        std    ="std",
        min    ="min",
        q1     =lambda s: s.quantile(0.25),
        median ="median",
        q3     =lambda s: s.quantile(0.75),
        max    ="max",
        n      ="count",
    )
    .reset_index()
    .sort_values("model")
)

print("\n[ORIGINAL PGD | 30 runs | Clean] base vs morph_ensemble:")
print(clean_orig_stats_30.to_string(index=False))

clean_orig_csv_30 = os.path.join(ORIG_PGD_RESULTS_DIR, "pgd_clean_stats_orig_30runs.csv")
clean_orig_stats_30.to_csv(clean_orig_csv_30, index=False)
print("Saved original PGD 30-run clean stats to:", clean_orig_csv_30)

df_pgd_orig_attack = df_orig_pgd[df_orig_pgd["attack"] == "pgd"].copy()
df_pgd_orig_attack = df_pgd_orig_attack[df_pgd_orig_attack["model"].isin(["base", "morph_rand"])]

def orig_pgd_model_stats(df, model_name, prefix="pgd_orig_30"):
    st = (
        df[df["model"] == model_name]
        .groupby("epsilon", observed=False)["RMSE"]
        .agg(
            mean   ="mean",
            std    ="std",
            min    ="min",
            q1     =lambda s: s.quantile(0.25),
            median ="median",
            q3     =lambda s: s.quantile(0.75),
            max    ="max",
            n      ="count",
        )
        .reset_index()
        .sort_values("epsilon")
    )

    print(f"\n[ORIGINAL PGD | 30 runs | PGD] {model_name} stats:")
    print(st.to_string(index=False))

    out_csv = os.path.join(ORIG_PGD_RESULTS_DIR, f"{prefix}_pgd_{model_name}_stats.csv")
    st.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    return st

orig_base_stats_30     = orig_pgd_model_stats(df_pgd_orig_attack, "base")
orig_morph_rand_stats_30 = orig_pgd_model_stats(df_pgd_orig_attack, "morph_rand")


[ORIGINAL PGD | 30 runs | Clean] base vs morph_ensemble:
         model     mean  std      min       q1   median       q3      max  n
          base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
morph_ensemble 0.191335  0.0 0.191335 0.191335 0.191335 0.191335 0.191335 30
Saved original PGD 30-run clean stats to: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results/pgd_clean_stats_orig_30runs.csv

[ORIGINAL PGD | 30 runs | PGD] base stats:
 epsilon     mean      std      min       q1   median       q3      max  n
     0.1 0.249214 0.000533 0.247921 0.248951 0.249227 0.249618 0.250336 30
     0.2 0.338651 0.000454 0.337487 0.338374 0.338667 0.338994 0.339499 30
     0.3 0.368207 0.000346 0.367480 0.368021 0.368237 0.368486 0.368956 30
     0.5 0.395957 0.000454 0.395118 0.395662 0.395977 0.396352 0.396755 30
Saved: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results/pgd_orig_30_pgd_base_stats.csv

[ORIGINAL PGD | 30 runs | PGD] morph_rand stats:
 epsilon     m

In [ ]:
COMBINED_PGD_DIR = NOV_PGD_RESULTS_DIR
os.makedirs(COMBINED_PGD_DIR, exist_ok=True)

# clean combined
df_clean_orig_comb = df_orig_pgd[df_orig_pgd["attack"] == "clean"].copy()
df_clean_nov_comb  = df_nov_pgd[df_nov_pgd["attack"] == "clean"].copy()

df_clean_orig_comb = df_clean_orig_comb[df_clean_orig_comb["model"].isin(["base", "morph_ensemble"])]
df_clean_nov_comb  = df_clean_nov_comb[df_clean_nov_comb["model"].isin(["nov_pgd_ensemble"])]

df_clean_comb = pd.concat([df_clean_orig_comb, df_clean_nov_comb], ignore_index=True)

df_clean_comb["model"] = pd.Categorical(
    df_clean_comb["model"],
    categories=["base", "morph_ensemble", "nov_pgd_ensemble"],
    ordered=True
)

pgd_clean_comp_stats_30 = (
    df_clean_comb
    .groupby("model", observed=False)["RMSE"]
    .agg(
        mean   ="mean",
        std    ="std",
        min    ="min",
        q1     =lambda s: s.quantile(0.25),
        median ="median",
        q3     =lambda s: s.quantile(0.75),
        max    ="max",
        n      ="count",
    )
    .reset_index()
    .sort_values("model")
)

print("\n[COMPARATIVE CLEAN | PGD | 30 runs] base vs morph_ensemble vs nov_pgd_ensemble:")
print(pgd_clean_comp_stats_30.to_string(index=False))

pgd_clean_comp_csv_30 = os.path.join(COMBINED_PGD_DIR, "pgd_clean_stats_orig_vs_nov_30runs.csv")
pgd_clean_comp_stats_30.to_csv(pgd_clean_comp_csv_30, index=False)
print("Saved comparative PGD clean stats (30 runs) to:", pgd_clean_comp_csv_30)

# PGD attack combined
df_pgd_orig_comb = df_orig_pgd[df_orig_pgd["attack"] == "pgd"].copy()
df_pgd_nov_comb  = df_nov_pgd[df_nov_pgd["attack"] == "pgd"].copy()

df_pgd_orig_comb = df_pgd_orig_comb[df_pgd_orig_comb["model"].isin(["base", "morph_rand"])]
df_pgd_nov_comb  = df_pgd_nov_comb[df_pgd_nov_comb["model"].isin(["nov_pgd_rand"])]

df_pgd_comb = pd.concat([df_pgd_orig_comb, df_pgd_nov_comb], ignore_index=True)

df_pgd_comb["model"] = pd.Categorical(
    df_pgd_comb["model"],
    categories=["base", "morph_rand", "nov_pgd_rand"],
    ordered=True
)

pgd_comp_stats_30 = (
    df_pgd_comb
    .groupby(["epsilon", "model"], observed=False)["RMSE"]
    .agg(
        mean   ="mean",
        std    ="std",
        min    ="min",
        q1     =lambda s: s.quantile(0.25),
        median ="median",
        q3     =lambda s: s.quantile(0.75),
        max    ="max",
        n      ="count",
    )
    .reset_index()
    .sort_values(["epsilon", "model"])
)

print("\n[COMPARATIVE PGD | 30 runs] base vs morph_rand vs nov_pgd_rand:")
print(pgd_comp_stats_30.to_string(index=False))

pgd_comp_csv_30 = os.path.join(COMBINED_PGD_DIR, "pgd_stats_orig_vs_nov_30runs.csv")
pgd_comp_stats_30.to_csv(pgd_comp_csv_30, index=False)
print("Saved comparative 30-run PGD stats to:", pgd_comp_csv_30)


[COMPARATIVE CLEAN | PGD | 30 runs] base vs morph_ensemble vs nov_pgd_ensemble:
           model     mean  std      min       q1   median       q3      max  n
            base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
  morph_ensemble 0.191335  0.0 0.191335 0.191335 0.191335 0.191335 0.191335 30
nov_pgd_ensemble 0.188289  0.0 0.188289 0.188289 0.188289 0.188289 0.188289 30
Saved comparative PGD clean stats (30 runs) to: /content/drive/MyDrive/298/Elec/NovTest/PGD/results/pgd_clean_stats_orig_vs_nov_30runs.csv

[COMPARATIVE PGD | 30 runs] base vs morph_rand vs nov_pgd_rand:
 epsilon        model     mean      std      min       q1   median       q3      max  n
     0.1         base 0.249214 0.000533 0.247921 0.248951 0.249227 0.249618 0.250336 30
     0.1   morph_rand 0.192592 0.000509 0.191700 0.192184 0.192627 0.192998 0.193406 30
     0.1 nov_pgd_rand 0.189792 0.000491 0.188747 0.189363 0.189839 0.190086 0.190914 30
     0.2         base 0.338651 0.000454 0.33748